In [ ]:
!pip -q install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from Bio import AlignIO
import numpy as np
import math
from collections import Counter
import random

# Parameters
MSA_FILE = "/kcnq_msa.fasta" # EDIT PATH, file is: kcnq_msa.fasta
KCNQ2_IDENTIFIER = "KCNQ2"
N_BOOTSTRAP = 1000
GAP_CHAR = "-"

PRINT_FIRST_N = 40
PRINT_LAST_N = 10

# Load MSA
alignment = AlignIO.read(MSA_FILE, "fasta")
seq_ids = [record.id for record in alignment]

kcnq2_index = None
for i, seq_id in enumerate(seq_ids):
    if KCNQ2_IDENTIFIER in seq_id:
        kcnq2_index = i
        break

if kcnq2_index is None:
    raise RuntimeError("KCNQ2 sequence not found in the MSA.")

print(f"KCNQ2 index in alignment: {kcnq2_index}")
print(f"Number of sequences: {len(alignment)}")
print(f"Alignment length: {alignment.get_alignment_length()}")

# Functions
def normalized_entropy(amino_acids):
    counts = Counter(amino_acids)
    total = sum(counts.values())
    if total <= 1:
        return 1.0

    probs = [c / total for c in counts.values()]
    entropy = -sum(p * math.log(p) for p in probs)
    max_entropy = math.log(len(counts))
    return 1.0 if max_entropy == 0 else (1.0 - entropy / max_entropy)

def bootstrap_conservation(column):
    aa_list = [aa for aa in column if aa != GAP_CHAR]
    if len(aa_list) <= 1:
        return [1.0] * N_BOOTSTRAP

    # bootstrap over the observed amino acids in this column
    return [
        normalized_entropy(random.choices(aa_list, k=len(aa_list)))
        for _ in range(N_BOOTSTRAP)
    ]

# Main
rows = []
kcnq2_pos = 0

for msa_pos in range(alignment.get_alignment_length()):
    col = alignment[:, msa_pos]
    if col[kcnq2_index] == GAP_CHAR:
        continue

    kcnq2_pos += 1
    boot = bootstrap_conservation(col)
    cons50 = float(np.percentile(boot, 50))
    cons95 = float(np.percentile(boot, 95))

    rows.append((msa_pos + 1, kcnq2_pos, cons50, cons95))

# Print
print("\nmsa_pos\tKCNQ2_pos\tcons50_kcnq\tcons95_kcnq\n")

for r in rows[:PRINT_FIRST_N]:
    print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}")

if PRINT_LAST_N > 0:
    print("\n--- LAST POSITIONS ---\n")
    for r in rows[-PRINT_LAST_N:]:
        print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}")

print("\n--- SUMMARY ---")
cons50_vals = [x[2] for x in rows]
cons95_vals = [x[3] for x in rows]
print(f"Total KCNQ2 positions: {len(rows)}")
print(f"cons50: min={min(cons50_vals):.4f}, median={np.median(cons50_vals):.4f}, max={max(cons50_vals):.4f}")
print(f"cons95: min={min(cons95_vals):.4f}, median={np.median(cons95_vals):.4f}, max={max(cons95_vals):.4f}")

KCNQ2 index in alignment: 0
Number of sequences: 5
Alignment length: 1120

msa_pos	KCNQ2_pos	cons50_kcnq	cons95_kcnq

1	1	1.000000	1.000000
2	2	0.081704	1.000000
3	3	0.081704	1.000000
4	4	0.081704	1.000000
5	5	0.081704	1.000000
6	6	0.081704	1.000000
21	7	0.081704	1.000000
22	8	1.000000	1.000000
28	9	1.000000	1.000000
29	10	0.081704	1.000000
30	11	0.188722	1.000000
31	12	0.188722	1.000000
32	13	0.135026	1.000000
33	14	0.039770	0.278072
34	15	0.029049	1.000000
35	16	0.029049	1.000000
36	17	0.039770	0.278072
37	18	0.039770	0.278072
45	19	0.135026	1.000000
46	20	0.135026	1.000000
47	21	0.029049	1.000000
48	22	0.039770	0.278072
49	23	1.000000	1.000000
50	24	0.039770	0.135026
51	25	0.039770	0.135026
52	26	0.135026	1.000000
57	27	0.135026	1.000000
58	28	0.135026	1.000000
59	29	0.039770	0.278072
60	30	0.135026	1.000000
68	31	0.039770	0.278072
69	32	0.135026	1.000000
70	33	0.039770	0.135026
71	34	0.039770	0.278072
72	35	0.039770	0.135026
73	36	0.039770	0.278072
74	37	0.135026	1.000000
75	38	1.0

In [ ]:
from Bio import AlignIO
import numpy as np
import math
from collections import Counter
import random

# Parameters
MSA_FILE = "/kcnq_msa.fasta" # EDIT PATH, file: kcnq_msa.fasta
KCNQ2_IDENTIFIER = "KCNQ2"
N_BOOTSTRAP = 1000
GAP_CHAR = "-"

# windows: radius (±r)
WIN3 = 3
WIN10 = 10

# how much to print
PRINT_FIRST_N = 40   # set to 999999 to print all
PRINT_LAST_N = 10    # set to 0 to disable

# Load MSA
alignment = AlignIO.read(MSA_FILE, "fasta")
seq_ids = [record.id for record in alignment]

kcnq2_index = None
for i, seq_id in enumerate(seq_ids):
    if KCNQ2_IDENTIFIER in seq_id:
        kcnq2_index = i
        break
if kcnq2_index is None:
    raise RuntimeError("KCNQ2 sequence not found in the MSA.")

print(f"KCNQ2 index in alignment: {kcnq2_index}")
print(f"Number of sequences: {len(alignment)}")
print(f"Alignment length: {alignment.get_alignment_length()}")

# Functions
def normalized_entropy(amino_acids):
    counts = Counter(amino_acids)
    total = sum(counts.values())
    if total <= 1:
        return 1.0
    probs = [c / total for c in counts.values()]
    entropy = -sum(p * math.log(p) for p in probs)
    max_entropy = math.log(20)
    return 1.0 - entropy / max_entropy

def bootstrap_by_sequences(column):
    m = len(column)
    vals = []
    for _ in range(N_BOOTSTRAP):
        idxs = [random.randrange(m) for _ in range(m)]
        sample_col = [column[i] for i in idxs]
        aa = [x for x in sample_col if x != GAP_CHAR]
        if len(aa) <= 1:
            vals.append(1.0)
        else:
            vals.append(normalized_entropy(aa))
    return vals

def moving_mean(arr, radius):
    n = len(arr)
    out = np.empty(n, dtype=float)
    for i in range(n):
        lo = max(0, i - radius)
        hi = min(n, i + radius + 1)
        out[i] = float(np.mean(arr[lo:hi]))
    return out

# MAIN (per-position)
msa_positions = []
kcnq2_positions = []
cons50 = []
cons95 = []

kcnq2_pos = 0
for msa_pos in range(alignment.get_alignment_length()):
    col = alignment[:, msa_pos]
    if col[kcnq2_index] == GAP_CHAR:
        continue
    kcnq2_pos += 1

    boot = bootstrap_by_sequences(col)
    c50 = float(np.percentile(boot, 50))
    c95 = float(np.percentile(boot, 95))

    msa_positions.append(msa_pos + 1)
    kcnq2_positions.append(kcnq2_pos)
    cons50.append(c50)
    cons95.append(c95)

cons50 = np.array(cons50, dtype=float)
cons95 = np.array(cons95, dtype=float)

# Smoothing windows
cons50_3aa  = moving_mean(cons50, WIN3)
cons95_3aa  = moving_mean(cons95, WIN3)
cons50_10aa = moving_mean(cons50, WIN10)
cons95_10aa = moving_mean(cons95, WIN10)

# Print
print("\nmsa_pos\tKCNQ2_pos\tcons50\tcons95\tcons50_3aa\tcons95_3aa\tcons50_10aa\tcons95_10aa\n")

rows = list(zip(
    msa_positions, kcnq2_positions,
    cons50, cons95,
    cons50_3aa, cons95_3aa,
    cons50_10aa, cons95_10aa
))

for r in rows[:PRINT_FIRST_N]:
    print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}\t{r[4]:.6f}\t{r[5]:.6f}\t{r[6]:.6f}\t{r[7]:.6f}")

if PRINT_LAST_N > 0:
    print("\n--- LAST POSITIONS ---\n")
    for r in rows[-PRINT_LAST_N:]:
        print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}\t{r[4]:.6f}\t{r[5]:.6f}\t{r[6]:.6f}\t{r[7]:.6f}")

print("\n--- SUMMARY ---")
print(f"Total KCNQ2 positions: {len(rows)}")
print(f"cons50 median={float(np.median(cons50)):.4f} | cons50_3aa median={float(np.median(cons50_3aa)):.4f} | cons50_10aa median={float(np.median(cons50_10aa)):.4f}")
print(f"cons95 median={float(np.median(cons95)):.4f} | cons95_3aa median={float(np.median(cons95_3aa)):.4f} | cons95_10aa median={float(np.median(cons95_10aa)):.4f}")

KCNQ2 index in alignment: 0
Number of sequences: 5
Alignment length: 1120

msa_pos	KCNQ2_pos	cons50	cons95	cons50_3aa	cons95_3aa	cons50_10aa	cons95_10aa

1	1	1.000000	1.000000	0.845312	1.000000	0.851673	1.000000
2	2	0.781435	1.000000	0.833755	1.000000	0.848391	1.000000
3	3	0.787526	1.000000	0.830177	1.000000	0.842772	1.000000
4	4	0.812288	1.000000	0.824084	1.000000	0.828850	0.988069
5	5	0.787526	1.000000	0.824084	1.000000	0.825282	0.988864
6	6	0.812288	1.000000	0.855308	1.000000	0.822161	0.989560
21	7	0.787526	1.000000	0.855308	1.000000	0.813963	0.980348
22	8	1.000000	1.000000	0.855308	1.000000	0.804735	0.972160
28	9	1.000000	1.000000	0.858845	1.000000	0.803188	0.973626
29	10	0.787526	1.000000	0.853567	1.000000	0.801796	0.974944
30	11	0.812288	1.000000	0.833615	0.976137	0.800536	0.976137
31	12	0.812288	1.000000	0.801521	0.976137	0.783768	0.968183
32	13	0.775343	1.000000	0.769427	0.976137	0.794175	0.968183
33	14	0.647859	0.832962	0.754465	0.952275	0.787525	0.957485
34	15	0.775343	1.0000

In [ ]:
from Bio import AlignIO
import numpy as np
import math, random
from collections import Counter

# ===================== 1) Inputs =====================
MSA_FILE = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"
KCNQ2_IDENTIFIER = "KCNQ2"
N_BOOTSTRAP = 400   # אפשר לעלות ל-1000 אם בא לך, 400 לרוב מספיק
GAP_CHAR = "-"

# --- Segmentation strength ---
# lambda גבוה => פחות חלונות (יותר "חלק")
# lambda נמוך => יותר חלונות (יותר "רגיש")
LAMBDA_SCALE = 0.6   # נסה 0.4–1.2 לפי הצורך (נראה בסוף כמה חלונות יצאו)

# --- Printing ---
PRINT_WINDOWS = True
PRINT_FIRST_N_POS = 40     # להדפיס עמדות ראשונות
PRINT_LAST_N_POS = 10      # להדפיס עמדות אחרונות
PRINT_ALL_POS = False      # אם True, מדפיס את כולן (הרבה)

# ===================== 2) Load MSA =====================
alignment = AlignIO.read(MSA_FILE, "fasta")
seq_ids = [rec.id for rec in alignment]

kcnq2_index = None
for i, sid in enumerate(seq_ids):
    if KCNQ2_IDENTIFIER in sid:
        kcnq2_index = i
        break
if kcnq2_index is None:
    raise RuntimeError("KCNQ2 sequence not found in the MSA.")

print(f"KCNQ2 index in alignment: {kcnq2_index}")
print(f"Number of sequences: {len(alignment)}")
print(f"Alignment length: {alignment.get_alignment_length()}")

# ===================== 3) Conservation per position =====================
def normalized_entropy_fixed20(amino_acids):
    # conservation = 1 - H / log(20)
    counts = Counter(amino_acids)
    total = sum(counts.values())
    if total <= 1:
        return 1.0
    probs = [c / total for c in counts.values()]
    H = -sum(p * math.log(p) for p in probs)
    return 1.0 - H / math.log(20)

def bootstrap_by_sequences(column, n_boot=400):
    # resample the paralogs (sequence dimension), like "resampling genes"
    m = len(column)
    vals = []
    for _ in range(n_boot):
        idxs = [random.randrange(m) for _ in range(m)]
        sample = [column[i] for i in idxs]
        aa = [x for x in sample if x != GAP_CHAR]
        if len(aa) <= 1:
            vals.append(1.0)
        else:
            vals.append(normalized_entropy_fixed20(aa))
    return vals

msa_pos_list = []
kcnq2_pos_list = []
cons50 = []
cons95 = []

kcnq2_pos = 0
for msa_pos in range(alignment.get_alignment_length()):
    col = alignment[:, msa_pos]
    if col[kcnq2_index] == GAP_CHAR:
        continue
    kcnq2_pos += 1
    boot = bootstrap_by_sequences(col, N_BOOTSTRAP)
    cons50.append(float(np.percentile(boot, 50)))
    cons95.append(float(np.percentile(boot, 95)))
    msa_pos_list.append(msa_pos + 1)
    kcnq2_pos_list.append(kcnq2_pos)

cons50 = np.array(cons50, dtype=float)
cons95 = np.array(cons95, dtype=float)
n = len(cons50)
print(f"\nTotal KCNQ2 positions (no gaps): {n}")

# ===================== 4) Piecewise-constant segmentation (DP) =====================
# We approximate their window model by finding contiguous segments where the score is ~constant.
# Minimize: sum_{segments} SSE(segment) + lambda * (#segments)

def segment_cost_prefix(prefix_sum, prefix_sq, i, j):
    # cost for [i, j) with mean-fit SSE
    # SSE = sum(y^2) - (sum(y)^2)/len
    s = prefix_sum[j] - prefix_sum[i]
    ss = prefix_sq[j] - prefix_sq[i]
    L = j - i
    return ss - (s * s) / L

def optimal_partition(y, lam):
    y = np.asarray(y, float)
    n = len(y)
    ps = np.zeros(n+1, float)
    ps2 = np.zeros(n+1, float)
    ps[1:] = np.cumsum(y)
    ps2[1:] = np.cumsum(y*y)

    F = np.full(n+1, np.inf, float)
    prev = np.full(n+1, -1, int)

    F[0] = -lam  # so the first segment also pays +lam (net 0 offset convenience)
    for t in range(1, n+1):
        best_val = np.inf
        best_s = -1
        # O(n^2) but n=872 -> fine
        for s in range(0, t):
            val = F[s] + segment_cost_prefix(ps, ps2, s, t) + lam
            if val < best_val:
                best_val = val
                best_s = s
        F[t] = best_val
        prev[t] = best_s

    # backtrack segments
    segs = []
    t = n
    while t > 0:
        s = prev[t]
        segs.append((s, t))  # [s, t)
        t = s
    segs.reverse()
    return segs

# choose lambda automatically from data scale
var = float(np.var(cons50))
lam = LAMBDA_SCALE * var * math.log(n + 1)
segs = optimal_partition(cons50, lam)

print(f"\nSegmentation lambda = {lam:.6g}  (scale={LAMBDA_SCALE})")
print(f"Number of windows = {len(segs)}")

# compute window "coeff" as mean within window, for both cons50 and cons95
win_starts = []
win_lengths = []
win_coeff50 = []
win_coeff95 = []

seg50 = np.zeros(n, float)
seg95 = np.zeros(n, float)

for (s, t) in segs:
    mean50 = float(np.mean(cons50[s:t]))
    mean95 = float(np.mean(cons95[s:t]))
    seg50[s:t] = mean50
    seg95[s:t] = mean95

    # convert to 1-based KCNQ2 positions
    start_pos = s + 1
    length = t - s
    win_starts.append(start_pos)
    win_lengths.append(length)
    win_coeff50.append(mean50)
    win_coeff95.append(mean95)

# ===================== 5) Print windows like C code =====================
if PRINT_WINDOWS:
    print("\n=== WINDOWS (like SelectionWindow: start, length, coeff) ===")
    print("win_id\tKCNQ2_start\tlength\tcoeff50\tcoeff95")
    for i, (st, ln, c50, c95) in enumerate(zip(win_starts, win_lengths, win_coeff50, win_coeff95), start=1):
        print(f"{i}\t{st}\t{ln}\t{c50:.6f}\t{c95:.6f}")

# ===================== 6) Print per-position values (raw + windowed) =====================
print("\n=== PER-POSITION (copy/paste friendly) ===")
print("KCNQ2_pos\tmsa_pos\tcons50_raw\tcons95_raw\tcons50_window\tcons95_window")

rows = list(zip(kcnq2_pos_list, msa_pos_list, cons50, cons95, seg50, seg95))

if PRINT_ALL_POS:
    for r in rows:
        print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}\t{r[4]:.6f}\t{r[5]:.6f}")
else:
    for r in rows[:PRINT_FIRST_N_POS]:
        print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}\t{r[4]:.6f}\t{r[5]:.6f}")

    if PRINT_LAST_N_POS > 0:
        print("\n--- LAST POSITIONS ---\n")
        for r in rows[-PRINT_LAST_N_POS:]:
            print(f"{r[0]}\t{r[1]}\t{r[2]:.6f}\t{r[3]:.6f}\t{r[4]:.6f}\t{r[5]:.6f}")

print("\n--- SUMMARY ---")
print(f"raw cons50 median={float(np.median(cons50)):.4f}, windowed median={float(np.median(seg50)):.4f}")
print(f"raw cons95 median={float(np.median(cons95)):.4f}, windowed median={float(np.median(seg95)):.4f}")

KCNQ2 index in alignment: 0
Number of sequences: 5
Alignment length: 1120

Total KCNQ2 positions (no gaps): 872

Segmentation lambda = 0.0598745  (scale=0.6)
Number of windows = 33

=== WINDOWS (like SelectionWindow: start, length, coeff) ===
win_id	KCNQ2_start	length	coeff50	coeff95
1	1	12	0.848591	1.000000
2	13	25	0.727783	0.910603
3	38	7	0.918968	1.000000
4	45	37	0.792965	0.947693
5	82	34	0.887122	0.985261
6	116	10	0.711601	0.877311
7	126	37	0.860913	0.977427
8	163	5	1.000000	1.000000
9	168	25	0.826884	0.973274
10	193	63	0.929123	0.994697
11	256	8	0.783123	0.951038
12	264	25	0.937514	0.986637
13	289	14	0.799077	0.952275
14	303	27	0.975254	1.000000
15	330	19	0.866163	0.961801
16	349	10	0.720515	0.908652
17	359	2	1.000000	1.000000
18	361	7	0.731404	0.928412
19	368	19	1.000000	1.000000
20	387	20	0.740620	0.940920
21	407	24	0.992179	1.000000
22	431	2	0.720230	0.916481
23	433	3	1.000000	1.000000
24	436	116	0.769122	0.946298
25	552	39	0.922144	0.995717
26	591	35	0.786912	0.960174
27	626	5

In [ ]:
import random, math
import numpy as np
from Bio import AlignIO

# ===================== INPUTS =====================
MSA_FILE = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"
KCNQ2_IDENTIFIER = "KCNQ2"
GAP = "-"

# MCMC params
N_ITERS   = 20000
BURN_IN   = 5000
THIN      = 10

# priors on coeff (Normal)
MU_COEFF  = 0.0
SIG_COEFF = 1.0

# move probabilities
P_SPLIT  = 0.25
P_MERGE  = 0.25
P_GROW   = 0.25
P_COEFF  = 0.25

# proposal step size for coeff changes
COEFF_STEP = 0.5

# ===================== LOAD MSA =====================
aln = AlignIO.read(MSA_FILE, "fasta")
ids = [r.id for r in aln]
kcnq2_idx = next((i for i,s in enumerate(ids) if KCNQ2_IDENTIFIER in s), None)
if kcnq2_idx is None:
    raise RuntimeError("KCNQ2 not found.")

L = aln.get_alignment_length()
S = len(aln)

# positions to model = columns where KCNQ2 is not gap
use_cols = []
for j in range(L):
    if aln[:, j][kcnq2_idx] != GAP:
        use_cols.append(j)

# Build a compact "protein columns" array for these positions:
# cols[p] is a list of length S (chars)
cols = [list(aln[:, j]) for j in use_cols]
N = len(cols)
print("Sequences:", S, "Alignment length:", L, "KCNQ2 positions:", N)

# Precompute per-position (m,k) ignoring gaps to speed likelihood
mk = []
for c in cols:
    aa = [x for x in c if x != GAP]
    k = len(aa)
    if k <= 1:
        mk.append((1, 1))  # treat as fully conserved / uninformative
    else:
        # mode count
        counts = {}
        for x in aa:
            counts[x] = counts.get(x, 0) + 1
        m = max(counts.values())
        mk.append((m, k))

# ===================== WINDOW MODEL =====================
# Window: start inclusive, length, coeff
def windows_to_lengths(wins):
    return [w["length"] for w in wins]

def check_windows(wins):
    assert wins[0]["start"] == 0
    for i in range(len(wins)-1):
        assert wins[i]["start"] + wins[i]["length"] == wins[i+1]["start"]
        assert wins[i]["length"] > 0
    assert wins[-1]["start"] + wins[-1]["length"] == N

def log_prior_coeff(coeff):
    # Normal(mu, sigma)
    return -0.5*((coeff-MU_COEFF)/SIG_COEFF)**2 - math.log(SIG_COEFF*math.sqrt(2*math.pi))

def log_lik_window(start, end, coeff):
    # end is exclusive
    beta = math.exp(coeff)  # strength >= 0
    ll = 0.0
    for i in range(start, end):
        m, k = mk[i]
        # penalty for mismatches, scaled by beta
        ll += beta * (m - k)  # <= 0
    return ll

def log_posterior(wins):
    check_windows(wins)
    lp = 0.0
    for w in wins:
        lp += log_prior_coeff(w["coeff"])
        lp += log_lik_window(w["start"], w["start"]+w["length"], w["coeff"])
    return lp

def pick_window_index(wins):
    return random.randrange(len(wins))

# ---- moves ----
def move_change_coeff(wins):
    w = pick_window_index(wins)
    old = wins[w]["coeff"]
    new = old + random.uniform(-COEFF_STEP, COEFF_STEP)
    wins2 = [dict(x) for x in wins]
    wins2[w]["coeff"] = new
    return wins2

def move_split(wins):
    # choose a window with length >=2
    candidates = [i for i,w in enumerate(wins) if w["length"] >= 2]
    if not candidates:
        return None
    wi = random.choice(candidates)
    w = wins[wi]
    # split point: between start+1 .. start+length-1
    cut = random.randrange(w["start"]+1, w["start"]+w["length"])
    left_len = cut - w["start"]
    right_len = w["length"] - left_len

    # propose new coeffs (simple symmetric proposal)
    c0 = w["coeff"] + random.uniform(-0.3, 0.3)
    c1 = w["coeff"] + random.uniform(-0.3, 0.3)

    wins2 = [dict(x) for x in wins]
    wins2[wi] = {"start": w["start"], "length": left_len, "coeff": c0}
    wins2.insert(wi+1, {"start": cut, "length": right_len, "coeff": c1})
    # fix downstream starts
    for j in range(wi+2, len(wins2)):
        wins2[j]["start"] = wins2[j-1]["start"] + wins2[j-1]["length"]
    return wins2

def move_merge(wins):
    if len(wins) <= 1:
        return None
    # pick boundary between window i-1 and i
    i = random.randrange(1, len(wins))
    wL, wR = wins[i-1], wins[i]
    new_len = wL["length"] + wR["length"]
    # new coeff: length-weighted average (simple)
    new_coeff = (wL["coeff"]*wL["length"] + wR["coeff"]*wR["length"]) / new_len

    wins2 = [dict(x) for x in wins]
    wins2[i-1] = {"start": wL["start"], "length": new_len, "coeff": new_coeff}
    del wins2[i]
    # fix downstream starts
    for j in range(i, len(wins2)):
        wins2[j]["start"] = wins2[j-1]["start"] + wins2[j-1]["length"]
    return wins2

def move_grow(wins):
    # move a boundary left/right by 1..d (small)
    if len(wins) <= 1:
        return None
    i = random.randrange(1, len(wins))  # boundary between i-1 and i
    wL, wR = wins[i-1], wins[i]
    if wL["length"] == 1 and wR["length"] == 1:
        return None

    step = random.choice([1,2,3])
    direction = random.choice([-1, +1])  # -1: move boundary left, +1: move right

    # moving boundary right means steal from right into left
    if direction == +1:
        if wR["length"] <= step:
            return None
        newL = wL["length"] + step
        newR = wR["length"] - step
    else:
        if wL["length"] <= step:
            return None
        newL = wL["length"] - step
        newR = wR["length"] + step

    wins2 = [dict(x) for x in wins]
    wins2[i-1]["length"] = newL
    wins2[i]["start"] = wins2[i-1]["start"] + newL
    wins2[i]["length"] = newR
    # fix downstream starts
    for j in range(i+1, len(wins2)):
        wins2[j]["start"] = wins2[j-1]["start"] + wins2[j-1]["length"]
    return wins2

def propose_move(wins):
    r = random.random()
    if r < P_SPLIT:
        return move_split(wins)
    r -= P_SPLIT
    if r < P_MERGE:
        return move_merge(wins)
    r -= P_MERGE
    if r < P_GROW:
        return move_grow(wins)
    return move_change_coeff(wins)

# ===================== RUN MCMC =====================
wins = [{"start": 0, "length": N, "coeff": 0.0}]  # start with 1 window
cur_lp = log_posterior(wins)

samples_per_pos = [[] for _ in range(N)]

accept = 0
for it in range(1, N_ITERS+1):
    prop = propose_move(wins)
    if prop is None:
        continue
    prop_lp = log_posterior(prop)

    # MH accept
    if math.log(random.random()) < (prop_lp - cur_lp):
        wins = prop
        cur_lp = prop_lp
        accept += 1

    # collect samples
    if it > BURN_IN and (it % THIN == 0):
        # assign coeff to each pos by its window
        for w in wins:
            st = w["start"]
            en = st + w["length"]
            c = w["coeff"]
            for p in range(st, en):
                samples_per_pos[p].append(c)

    if it % 2000 == 0:
        print(f"iter {it} | windows={len(wins)} | accept_rate~{accept/it:.3f}")

# ===================== OUTPUT percentiles =====================
def pct(x, q):
    return float(np.percentile(x, q)) if len(x) else float("nan")

print("\nKCNQ2_pos\tcoeff50\tcoeff95")
for p in range(0, min(40, N)):
    s = samples_per_pos[p]
    print(f"{p+1}\t{pct(s,50):.6f}\t{pct(s,95):.6f}")

print("\n--- LAST ---")
for p in range(max(0, N-10), N):
    s = samples_per_pos[p]
    print(f"{p+1}\t{pct(s,50):.6f}\t{pct(s,95):.6f}")

Sequences: 5 Alignment length: 1120 KCNQ2 positions: 872
iter 8000 | windows=1 | accept_rate~0.201
iter 10000 | windows=1 | accept_rate~0.199
iter 14000 | windows=1 | accept_rate~0.197

KCNQ2_pos	coeff50	coeff95
1	-5.411270	-4.817159
2	-5.411270	-4.817159
3	-5.411270	-4.817159
4	-5.411270	-4.817159
5	-5.411270	-4.817159
6	-5.411270	-4.817159
7	-5.411270	-4.817159
8	-5.411270	-4.817159
9	-5.411270	-4.817159
10	-5.411270	-4.817159
11	-5.411270	-4.817159
12	-5.411270	-4.817159
13	-5.411270	-4.817159
14	-5.411270	-4.817159
15	-5.411270	-4.817159
16	-5.411270	-4.817159
17	-5.411270	-4.817159
18	-5.411270	-4.817159
19	-5.411270	-4.817159
20	-5.411270	-4.817159
21	-5.411270	-4.817159
22	-5.411270	-4.817159
23	-5.411270	-4.817159
24	-5.411270	-4.817159
25	-5.411270	-4.817159
26	-5.411270	-4.817159
27	-5.411270	-4.817159
28	-5.411270	-4.817159
29	-5.411270	-4.817159
30	-5.411270	-4.817159
31	-5.411270	-4.817159
32	-5.411270	-4.817159
33	-5.411270	-4.817159
34	-5.411270	-4.817159
35	-5.411270	-4

# **WITH CDS SEQUENCES**

In [ ]:
# ===================== 0) Mount Google Drive =====================
from google.colab import drive
drive.mount("/content/drive")

# ===================== 1) Paths =====================
# Your CDS folder (Drive path in Colab form)
CDS_DIR = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ CDS from Ensembl"

# Your existing protein MSA (you already have this)
PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

print("CDS_DIR:", CDS_DIR)
print("PROT_MSA:", PROT_MSA)

# ===================== 2) List CDS files and concatenate =====================
import glob, os, textwrap

cds_files = sorted(glob.glob(os.path.join(CDS_DIR, "*.fasta")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fa")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fna")))

print("\nFound CDS files:")
for f in cds_files:
    print(" -", os.path.basename(f))

if len(cds_files) == 0:
    raise FileNotFoundError("No CDS FASTA files found in CDS_DIR (expected .fasta/.fa/.fna).")

# Concatenate into one file
CDS_ALL = "/content/kcnq_cds_all.fasta"
with open(CDS_ALL, "w") as out:
    for f in cds_files:
        with open(f, "r") as inp:
            out.write(inp.read().strip() + "\n")

print("\nWrote concatenated CDS:", CDS_ALL)
print("Preview:")
print("".join(open(CDS_ALL).read().splitlines(True)[:12]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CDS_DIR: /content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ CDS from Ensembl
PROT_MSA: /content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta

Found CDS files:
 - Homo_sapiens_KCNQ1_sequence.fa
 - Homo_sapiens_KCNQ2_sequence.fa
 - Homo_sapiens_KCNQ3_sequence.fa
 - Homo_sapiens_KCNQ4_sequence.fa
 - Homo_sapiens_KCNQ5_sequence.fa

Wrote concatenated CDS: /content/kcnq_cds_all.fasta
Preview:
>KCNQ1-212 cds:protein_coding
NNGCTGCGGGAACACCATCGGGCCACCATTAAGGTCATTCGACGCATGCAGTACTTTGTG
GCCAAGAAGAAATTCCAGCAAGCGCGGAAGCCTTACGATGTGCGGGACGTCATTGAGCAG
TACTCGCAGGGCCACCTCAACCTCATGGTGCGCATCAAGGAGCTGCAGAGGAGGCTGGAC
CAGTCCATTGGGAAGCCCTCACTGTTCATCTCCGTCTCAGAAAAGAGCAAGGATCGCGGC
AGCAACACGATCGGCGCCCGCCTGAACCGAGTAGAAGACAAGGTGACGCAGCTGGACCAG
AGGCTGGCACTCATCACCGACATGCTTCACCAGCTGCTCTCCTTGCACGGTGGCAGCACC
CCCGGCAGCGGCGGCCCCCCCAGAGAGGGCGGGGCCCACATCACCCAGCCCTGCGGCAGT
GGCGGCTCCGT

In [ ]:
# ===================== 3) Translate CDS -> protein (sanity checks) =====================
!pip -q install biopython

from Bio import SeqIO
from Bio.Seq import Seq

records = list(SeqIO.parse(CDS_ALL, "fasta"))

bad = []
prot_out = []

def clean_nt(seq):
    s = str(seq).upper().replace(" ", "").replace("\n", "").replace("\r", "")
    # keep only A/C/G/T/N
    return "".join([c for c in s if c in "ACGTN"])

for r in records:
    s = clean_nt(r.seq)

    if len(s) % 3 != 0:
        bad.append((r.id, "length_not_multiple_of_3", len(s)))

    aa = str(Seq(s).translate(to_stop=False))
    # internal stop codons are suspicious (allow possibly last stop)
    if "*" in aa[:-1]:
        bad.append((r.id, "internal_stop_codon", aa.count("*")))

    prot_out.append((r.id, aa))

print("Total CDS sequences:", len(records))
print("Problems found:", len(bad))
for x in bad[:15]:
    print("  ", x)

# Write translated proteins (useful for ID matching/debug)
CDS_PROT = "/content/kcnq_cds_translated.fasta"
with open(CDS_PROT, "w") as f:
    for rid, aa in prot_out:
        f.write(f">{rid}\n{aa}\n")

print("\nWrote translated proteins:", CDS_PROT)
print("Preview:")
print("".join(open(CDS_PROT).read().splitlines(True)[:12]))

Total CDS sequences: 83
Problems found: 1
   ('KCNQ2-226', 'length_not_multiple_of_3', 152)

Wrote translated proteins: /content/kcnq_cds_translated.fasta
Preview:
>KCNQ1-212
XLREHHRATIKVIRRMQYFVAKKKFQQARKPYDVRDVIEQYSQGHLNLMVRIKELQRRLDQSIGKPSLFISVSEKSKDRGSNTIGARLNRVEDKVTQLDQRLALITDMLHQLLSLHGGSTPGSGGPPREGGAHITQPCGSGGSVDPELFLPSNTLPTYEQLTVPRRGPDEGS*
>KCNQ1-211
XLREHHRATIKVIRRMQYFVAKKKFQQARKPYDVRDVIEQYSQGHLNLMVRIKELQRRLDQSIGKPSLFISVSEKSKDRGSNTIGARLNRVEDKVTQLDQRLALITDMLHQLLSLHGGSTPGSGGPPREGGAHITQPCGSGGSVDPELFLPSNTLPTYEQLTVPRRGPDEGS*
>KCNQ1-213

>KCNQ2-204
MVQKSRNGGVYPGPSGEKKLKVGFVGLDPGAPDSTRDGALLIAGSEAPKRGSILSKPRAGGAGAGKPPKRNAFYRKLQNFLYNVLERPRGWAFIYHAYVFLLVFSCLVLSVFSTIKEYEKSSEGALYILEIVTIVVFGVEYFVRIWAAGCCCRYRGWRGRLKFARKPFCVIDIMVLIASIAVLAAGSQGNVFATSALRSLRFLQILRMIRMDRRGGTWKLLGSVVYAHSKELVTAWYIGFLCLILASFLVYLAEKGENDHFDTYADALWWGLITLTTIGYGDKYPQTWNGRLLAATFTLIGVSFFALPAGILGSGFALKVQEQHRQKHFEKRRNPAAGLIQSAWRFYATNLSRTDLHSTWQYYERTVTVPMYSSQTQTYGASRLIPPLNQLELLRNLKSKSGLAFRKDPPPEPSPSKGSPCRGPLCGCCPGRSSQKVSLKDRV

/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [ ]:
# ===================== 4) Create codon alignment with pal2nal =====================
# We’ll download pal2nal (perl script) and run it:
!apt-get -qq update
!apt-get -qq install -y perl wget

# Download pal2nal v14 tarball from EMBL Bork site (classic source).
# If this fails in your environment, see fallback cell below.
!wget -q -O /content/pal2nal.v14.tar.gz http://www.bork.embl.de/pal2nal/distribution/pal2nal.v14.tar.gz

# Extract
!tar -xzf /content/pal2nal.v14.tar.gz -C /content
!ls /content/pal2nal.v14 | head

# Run pal2nal:
# Input 1: protein MSA
# Input 2: CDS fasta (concatenated, unaligned)
# Output: codon alignment fasta
CODON_MSA = "/content/kcnq_codon_msa.fasta"

!perl /content/pal2nal.v14/pal2nal.pl "$PROT_MSA" "$CDS_ALL" -output fasta > "$CODON_MSA"

print("Wrote codon MSA:", CODON_MSA)
print("Preview:")
print("".join(open(CODON_MSA).read().splitlines(True)[:12]))

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
for_paml
pal2nal.pl
README
test.aln
test.nuc

ERROR: number of input seqs differ (aa: 5;  nuc: 83)!!

   aa  'sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2 sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_4_OS_Homo_sapiens_OX_9606_GN_KCNQ4_PE_1_SV_2 sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_5_OS_Homo_sapiens_OX_9606_GN_KCNQ5_PE_1_SV_3 sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_3_OS_Homo_sapiens_OX_9606_GN_KCNQ3_PE_1_SV_2 sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3'
   nuc 'KCNQ1-212 KCNQ1-211 KCNQ1-213 KCNQ2-204 KCNQ2-214 KCNQ2-238 KCNQ2-245 KCNQ2-213 KCNQ2-209 KC

In [ ]:
# ===================== 5) Sanity check codon MSA =====================
from Bio import SeqIO

recs = list(SeqIO.parse(CODON_MSA, "fasta"))
lens = sorted({len(r.seq) for r in recs})

print("Number of sequences in codon MSA:", len(recs))
print("Unique lengths:", lens)
print("All lengths multiple of 3?", all(len(r.seq) % 3 == 0 for r in recs))

# Also check for gaps count
gap_counts = [(r.id, str(r.seq).count("-")) for r in recs]
gap_counts_sorted = sorted(gap_counts, key=lambda x: x[1], reverse=True)
print("\nTop gap counts:")
for gid, gc in gap_counts_sorted[:5]:
    print(f"  {gid}: {gc} gaps")

Number of sequences in codon MSA: 0
Unique lengths: []
All lengths multiple of 3? True

Top gap counts:


# **ATTEMPT 2**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install biopython

import os, glob, math
from Bio import SeqIO
from Bio.Seq import Seq

CDS_DIR = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ CDS from Ensembl"
PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

# ---------------- helpers ----------------
def clean_nt(seq):
    s = str(seq).upper().replace(" ", "").replace("\n", "").replace("\r", "")
    return "".join([c for c in s if c in "ACGTN"])

def translate_cds(nt):
    nt = clean_nt(nt)
    # make length multiple of 3 by trimming trailing partial codon (safe for Ensembl dumps)
    nt = nt[: (len(nt)//3)*3]
    aa = str(Seq(nt).translate(to_stop=False))
    # drop trailing stop if exists
    if aa.endswith("*"):
        aa = aa[:-1]
    return aa

def ungap_protein(seq):
    return str(seq).replace("-", "").replace(".", "")

def gene_from_cds_id(cds_id):
    # cds_id like "KCNQ2-204" or "KCNQ1-212"
    return cds_id.split("-")[0]

def gene_from_msa_id(msa_id):
    # msa_id contains "..._KCNQ2_HUMAN_..."
    for g in ["KCNQ1","KCNQ2","KCNQ3","KCNQ4","KCNQ5"]:
        if g in msa_id:
            return g
    return None

def identity(a, b):
    # compare up to min length (they should match exactly in our best case)
    L = min(len(a), len(b))
    if L == 0:
        return 0.0
    return sum(1 for i in range(L) if a[i] == b[i]) / L

# ---------------- read protein MSA (targets) ----------------
msa_records = list(SeqIO.parse(PROT_MSA, "fasta"))
targets = {}   # gene -> (msa_id, protein_seq_ungapped)
for r in msa_records:
    g = gene_from_msa_id(r.id)
    if g is None:
        raise RuntimeError(f"Could not infer gene from MSA id: {r.id}")
    targets[g] = (r.id, ungap_protein(r.seq))

print("Targets from protein MSA:")
for g,(mid,prot) in targets.items():
    print(f"  {g}: id='{mid}' len={len(prot)}")

# ---------------- read all CDS records (many isoforms) ----------------
cds_files = sorted(glob.glob(os.path.join(CDS_DIR, "*.fa")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fasta")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fna")))

all_cds = []
for f in cds_files:
    all_cds.extend(list(SeqIO.parse(f, "fasta")))

print("\nTotal CDS records found:", len(all_cds))

# group by gene
cds_by_gene = {g: [] for g in targets.keys()}
for r in all_cds:
    g = gene_from_cds_id(r.id)
    if g in cds_by_gene:
        cds_by_gene[g].append(r)

for g in targets.keys():
    print(f"{g}: {len(cds_by_gene[g])} CDS isoforms")

# ---------------- match best CDS per gene ----------------
chosen = {}  # gene -> (msa_id, cds_record, aa_translated, score)
for g, (msa_id, prot_target) in targets.items():
    best = None
    for cds_rec in cds_by_gene[g]:
        aa = translate_cds(cds_rec.seq)
        # primary: exact match length + sequence
        if aa == prot_target:
            best = (msa_id, cds_rec, aa, (1.0, 0))  # perfect
            break

        # fallback score: identity + length penalty
        idn = identity(aa, prot_target)
        len_pen = abs(len(aa) - len(prot_target))
        score = (idn, -len_pen)
        if best is None or score > best[3]:
            best = (msa_id, cds_rec, aa, score)

    if best is None:
        raise RuntimeError(f"No CDS candidates found for {g}")
    chosen[g] = best

print("\nChosen isoforms:")
for g,(msa_id, cds_rec, aa, score) in chosen.items():
    print(f"  {g}: CDS='{cds_rec.id}' aa_len={len(aa)} target_len={len(targets[g][1])} score={score}")

# ---------------- write filtered CDS fasta with IDs matching MSA ----------------
OUT_CDS_5 = "/content/kcnq_cds_5_matched.fasta"
with open(OUT_CDS_5, "w") as f:
    for g,(msa_id, cds_rec, aa, score) in chosen.items():
        nt = clean_nt(cds_rec.seq)
        nt = nt[: (len(nt)//3)*3]
        # IMPORTANT: header must match protein MSA record.id
        f.write(f">{msa_id}\n{nt}\n")

print("\nWrote 5-seq CDS file with MSA-matching headers:")
print(" ", OUT_CDS_5)
print("Preview:")
print("".join(open(OUT_CDS_5).read().splitlines(True)[:12]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Targets from protein MSA:
  KCNQ2: id='sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2' len=872
  KCNQ4: id='sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_4_OS_Homo_sapiens_OX_9606_GN_KCNQ4_PE_1_SV_2' len=695
  KCNQ5: id='sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_5_OS_Homo_sapiens_OX_9606_GN_KCNQ5_PE_1_SV_3' len=932
  KCNQ3: id='sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_3_OS_Homo_sapiens_OX_9606_GN_KCNQ3_PE_1_SV_2' len=872
  KCNQ1: id='sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3' len=676

Total CDS records found: 83
KCNQ2: 53 CDS isoforms
KCNQ4: 6 CDS isoforms
KCNQ5: 14 CDS isoforms
KCNQ3: 7 CDS isoforms
KCNQ1: 3 CDS iso

In [ ]:
!apt-get -qq update
!apt-get -qq install -y perl wget

# download + extract pal2nal v14
!wget -q -O /content/pal2nal.v14.tar.gz http://www.bork.embl.de/pal2nal/distribution/pal2nal.v14.tar.gz
!tar -xzf /content/pal2nal.v14.tar.gz -C /content

CODON_MSA = "/content/kcnq_codon_msa.fasta"
!perl /content/pal2nal.v14/pal2nal.pl "$PROT_MSA" "/content/kcnq_cds_5_matched.fasta" -output fasta > "$CODON_MSA"

print("Wrote codon MSA:", CODON_MSA)
print("Preview:")
print("".join(open(CODON_MSA).read().splitlines(True)[:12]))

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
#---  ERROR: inconsistency between the following pep and nuc seqs  ---#
>sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3
MAAASSPPRAERKRWGWGRLPGARRGSAGLAKKCPFSLELAEGGPAGGALYAPIAPGAPG
PAPPASPAAPAAPPVASDLGPRPPVSLDPRVSIYSTRRPVLARTHVQGRVYNFLERPTGW
KCFVYHFAVFLIVLVCLIFSVLSTIEQYAALATGTLFWMEIVLVVFFGTEYVVRLWSAGC
RSKYVGLWGRLRFARKPISIIDLIVVVASMVVLCVGSKGQVFATSAIRGIRFLQILRMLH
VDRQGGTWRLLGSVVFIHRQELITTLYIGFLGLIFSSYFVYLAEKDAVNESGRVEFGSYA
DALWWGVVTVTTIGYGDKVPQTWVGKTIASCFSVFAISFFALPAGILGSGFALKVQQKQR
QKHFNRQIPAAASLIQTAWRCYAAENPDSSTWKIYIRKAPRSHTLLSPSPKPKKSVVVKK
KKFKLDKDNGVTPGEKMLTVPHITCDPPEERRLDHFSVDGYDSSVRKSPTLLEVSMPHFM
RTNSFAEDLDLEGETLLTPITHISQLREHHRATIKVIRRMQYFVAKKKFQQARKPYDVRD
VIEQYSQGHLNLMVRIKELQRRLDQSIGKPSLFISVSEKSKDRGSNTIGARLNRVEDKVT
QLDQRLAL

In [ ]:
from Bio import SeqIO
recs = list(SeqIO.parse("/content/kcnq_codon_msa.fasta", "fasta"))
print("Number of sequences:", len(recs))
lens = {len(r.seq) for r in recs}
print("Unique lengths:", lens)
print("All multiple of 3?", all(len(r.seq)%3==0 for r in recs))

Number of sequences: 0
Unique lengths: set()
All multiple of 3? True


attempt 3

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install biopython

import os, glob
from Bio import SeqIO
from Bio.Seq import Seq

CDS_DIR = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ CDS from Ensembl"
PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"
GAP="-"

GENES = ["KCNQ1","KCNQ2","KCNQ3","KCNQ4","KCNQ5"]

def gene_from_msa_id(msa_id):
    for g in GENES:
        if g in msa_id:
            return g
    return None

def gene_from_cds_id(cds_id):
    # like KCNQ1-212
    return cds_id.split("-")[0]

def clean_nt(seq):
    s = str(seq).upper().replace(" ", "").replace("\n", "").replace("\r", "")
    return "".join([c for c in s if c in "ACGTN"])

def translate_nt(nt):
    nt = clean_nt(nt)
    nt = nt[: (len(nt)//3)*3]
    aa = str(Seq(nt).translate(to_stop=False))
    if aa.endswith("*"):
        aa = aa[:-1]
    return aa

def ungap_protein(seq):
    return str(seq).replace("-", "").replace(".", "")

def try_fix_by_atg(nt, target_aa):
    """
    Try to rescue CDS by cutting to an ATG start in-frame.
    Return nt_fixed if translates exactly to target_aa, else None.
    """
    nt = clean_nt(nt)
    # scan ATG positions; keep only those with length multiple of 3
    for start in range(0, len(nt)-2):
        if nt[start:start+3] == "ATG":
            nt2 = nt[start:]
            nt2 = nt2[: (len(nt2)//3)*3]
            aa2 = translate_nt(nt2)
            if aa2 == target_aa:
                return nt2
    return None

# ---------- read protein targets from MSA ----------
msa_records = list(SeqIO.parse(PROT_MSA, "fasta"))
targets = {}  # gene -> (msa_id, aa_ungapped)
for r in msa_records:
    g = gene_from_msa_id(r.id)
    if g is None:
        raise RuntimeError(f"Can't infer gene from MSA id: {r.id}")
    targets[g] = (r.id, ungap_protein(r.seq))

print("Targets (from protein MSA):")
for g,(mid,aa) in targets.items():
    print(g, "len", len(aa), "|", mid[:60]+"...")

# ---------- load all CDS records ----------
cds_files = sorted(glob.glob(os.path.join(CDS_DIR, "*.fa")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fasta")) +
                   glob.glob(os.path.join(CDS_DIR, "*.fna")))
all_cds = []
for f in cds_files:
    all_cds.extend(list(SeqIO.parse(f, "fasta")))

print("\nTotal CDS records:", len(all_cds))

cds_by_gene = {g: [] for g in GENES}
for r in all_cds:
    g = gene_from_cds_id(r.id)
    if g in cds_by_gene:
        cds_by_gene[g].append(r)

for g in GENES:
    print(g, "CDS isoforms:", len(cds_by_gene[g]))

# ---------- choose EXACT matches only (with ATG-rescue) ----------
chosen = {}  # gene -> (msa_id, cds_id, nt_used, mode)
for g in GENES:
    msa_id, target_aa = targets[g]
    found = None

    # pass 1: exact match as-is
    for rec in cds_by_gene[g]:
        aa = translate_nt(rec.seq)
        if aa == target_aa:
            found = ("exact", rec.id, clean_nt(rec.seq))
            break

    # pass 2: try ATG rescue (cut to in-frame ATG)
    if found is None:
        for rec in cds_by_gene[g]:
            nt_fixed = try_fix_by_atg(rec.seq, target_aa)
            if nt_fixed is not None:
                found = ("atg_rescue", rec.id, nt_fixed)
                break

    if found is None:
        print(f"\n❌ No EXACT CDS match found for {g} (even with ATG rescue).")
        # Print top 5 closest by length to help diagnose
        cand = []
        for rec in cds_by_gene[g]:
            aa = translate_nt(rec.seq)
            cand.append((abs(len(aa)-len(target_aa)), rec.id, len(aa), aa[:15]))
        cand.sort()
        print("Closest-by-length candidates:")
        for x in cand[:5]:
            print("  ", x)
        raise RuntimeError(f"Stop: need a matching CDS for {g}.")

    mode, cds_id, nt_used = found
    chosen[g] = (msa_id, cds_id, nt_used, mode)

print("\n✅ Chosen CDS (exactly matching MSA proteins):")
for g,(msa_id, cds_id, nt_used, mode) in chosen.items():
    print(f"  {g}: {mode} | {cds_id} | nt_len={len(nt_used)} | header={msa_id[:40]}...")

# ---------- write 5-seq CDS with headers MATCHING MSA ----------
OUT_CDS_5 = "/content/kcnq_cds_5_matched.fasta"
with open(OUT_CDS_5, "w") as f:
    for g in GENES:
        msa_id, cds_id, nt_used, mode = chosen[g]
        nt_used = nt_used[: (len(nt_used)//3)*3]
        f.write(f">{msa_id}\n{nt_used}\n")

print("\nWrote:", OUT_CDS_5)
print("Preview:\n", "".join(open(OUT_CDS_5).read().splitlines(True)[:8]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Targets (from protein MSA):
KCNQ2 len 872 | sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ4 len 695 | sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ5 len 932 | sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ3 len 872 | sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ1 len 676 | sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfam...

Total CDS records: 83
KCNQ1 CDS isoforms: 3
KCNQ2 CDS isoforms: 53
KCNQ3 CDS isoforms: 7
KCNQ4 CDS isoforms: 6
KCNQ5 CDS isoforms: 14

❌ No EXACT CDS match found for KCNQ1 (even with ATG rescue).
Closest-by-length candidates:
   (504, 'KCNQ1-211', 172, 'XLREHHRATIKVIRR')
   (504, 'KCNQ1-212', 172, 'XLREHHRATIKVIRR')
   (676, 'KCNQ1-213', 0, '')


RuntimeError: Stop: need a matching CDS for KCNQ1.

new attempt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install biopython requests

import requests
from Bio import SeqIO
from Bio.Seq import Seq

PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

# 1) ENSTs (MANE/representative)
ENST = {
    "KCNQ1": "ENST00000155840",  # MANE Select (Ensembl page)
    "KCNQ2": "ENST00000359125",  # KCNQ2-204 (872 aa)
    "KCNQ3": "ENST00000521134",  # KCNQ3-204 (we'll validate vs MSA)
    "KCNQ4": "ENST00000347132",  # MANE Select
    "KCNQ5": "ENST00000370398",  # MANE Select/default transcript on gene page
}

def ungap(s):
    return s.replace("-", "").replace(".", "")

# 2) load protein targets from your protein MSA
targets = {}  # gene -> (msa_id, aa)
for r in SeqIO.parse(PROT_MSA, "fasta"):
    for g in ENST.keys():
        if g in r.id:
            targets[g] = (r.id, ungap(str(r.seq)))
            break

print("Protein targets loaded from MSA:")
for g,(mid,aa) in targets.items():
    print(g, "len", len(aa), "|", mid[:60]+"...")

# 3) fetch CDS for each ENST from Ensembl REST
def fetch_cds(enst_id):
    url = f"https://rest.ensembl.org/sequence/id/{enst_id}?type=cds"
    r = requests.get(url, headers={"Content-Type":"text/plain"})
    r.raise_for_status()
    seq = r.text.strip().replace("\n","").replace("\r","")
    return seq.upper()

def translate_cds(nt):
    nt = "".join([c for c in nt if c in "ACGTN"])
    nt = nt[: (len(nt)//3)*3]
    aa = str(Seq(nt).translate(to_stop=False))
    if aa.endswith("*"):
        aa = aa[:-1]
    return aa

chosen_cds = {}  # gene -> (msa_id, cds_nt)
for g,enst_id in ENST.items():
    cds = fetch_cds(enst_id)
    aa = translate_cds(cds)
    msa_id, target_aa = targets[g]

    print(f"\n{g}: ENST={enst_id} CDS_len={len(cds)} AA_len={len(aa)} target_len={len(target_aa)}")
    if aa != target_aa:
        # print small diagnostic
        print("❌ AA mismatch vs your protein MSA.")
        print("AA start:", aa[:30])
        print("Target :", target_aa[:30])
        raise RuntimeError(f"Stop: {g} ENST protein does not match your protein MSA. Need correct transcript for this gene.")
    else:
        print("✅ Perfect match to protein MSA.")
        chosen_cds[g] = (msa_id, cds)

# 4) write the 5 CDS with headers matching the protein MSA IDs
OUT_CDS_5 = "/content/kcnq_cds_5_matched.fasta"
with open(OUT_CDS_5, "w") as f:
    for g,(msa_id, cds) in chosen_cds.items():
        f.write(f">{msa_id}\n{cds}\n")

print("\nWrote:", OUT_CDS_5)
print("Preview:")
print("".join(open(OUT_CDS_5).read().splitlines(True)[:8]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Protein targets loaded from MSA:
KCNQ2 len 872 | sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ4 len 695 | sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ5 len 932 | sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ3 len 872 | sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfam...
KCNQ1 len 676 | sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfam...

KCNQ1: ENST=ENST00000155840 CDS_len=2031 AA_len=676 target_len=676
✅ Perfect match to protein MSA.

KCNQ2: ENST=ENST00000359125 CDS_len=2619 AA_len=872 target_len=872
✅ Perfect match to protein MSA.

KCNQ3: ENST=ENST00000521134 CDS_len=2259 AA_len=752 target_len=872
❌ AA mismatch vs your protein MSA.
AA start: MKPAEHATMFLIVLGCLILAVLTTFKEYET
Target : MGLKARRAAGAAGGGGDGGGGGGGAANPAG


RuntimeError: Stop: KCNQ3 ENST protein does not match your protein MSA. Need correct transcript for this gene.

In [ ]:
import requests
from Bio import SeqIO

PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

GENE = "KCNQ3"

def ungap(s):
    return s.replace("-", "").replace(".", "")

# target AA from your protein MSA
target_msa_id = None
target_aa = None
for r in SeqIO.parse(PROT_MSA, "fasta"):
    if GENE in r.id:
        target_msa_id = r.id
        target_aa = ungap(str(r.seq))
        break

print("Target:", GENE, "len", len(target_aa))
print("MSA id:", target_msa_id)

# get gene model + transcripts
lookup_url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{GENE}?expand=1"
gene_info = requests.get(lookup_url, headers={"Content-Type":"application/json"}).json()

transcripts = gene_info.get("Transcript", [])
print("Transcripts found:", len(transcripts))

def fetch_protein_by_translation_id(tr_id):
    url = f"https://rest.ensembl.org/sequence/id/{tr_id}?type=protein"
    r = requests.get(url, headers={"Content-Type":"text/plain"})
    if r.status_code != 200:
        return None
    return r.text.strip().replace("\n","").replace("\r","")

def fetch_cds_by_enst(enst):
    url = f"https://rest.ensembl.org/sequence/id/{enst}?type=cds"
    r = requests.get(url, headers={"Content-Type":"text/plain"})
    r.raise_for_status()
    return r.text.strip().replace("\n","").replace("\r","").upper()

best = None  # (is_exact, length_diff, enst, translation_id, aa)
for t in transcripts:
    enst = t.get("id")
    tr = t.get("Translation")
    if not enst or not tr or not tr.get("id"):
        continue
    translation_id = tr["id"]
    aa = fetch_protein_by_translation_id(translation_id)
    if aa is None:
        continue

    is_exact = (aa == target_aa)
    length_diff = abs(len(aa) - len(target_aa))

    cand = (is_exact, -length_diff, enst, translation_id, aa)
    if best is None or cand > best:
        best = cand

    if is_exact:
        print("✅ FOUND exact match!")
        print("ENST:", enst)
        print("Translation:", translation_id)
        print("AA len:", len(aa))
        break

if best is None:
    raise RuntimeError("Could not retrieve any translations for KCNQ3.")

if not best[0]:
    # no exact match; show best near-match info
    _, neg_len_diff, enst, translation_id, aa = best
    print("\n❌ No exact match found. Best near-match:")
    print("ENST:", enst, "| Translation:", translation_id)
    print("AA len:", len(aa), "| target len:", len(target_aa))
    print("AA start   :", aa[:40])
    print("Target start:", target_aa[:40])
    raise RuntimeError("Stop: need correct transcript (exact protein match).")

# if exact, fetch CDS
_, _, enst, translation_id, aa = best
kcnq3_cds = fetch_cds_by_enst(enst)
print("\nFetched CDS for matched KCNQ3 ENST:", enst, "CDS len:", len(kcnq3_cds))

Target: KCNQ3 len 872
MSA id: sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_3_OS_Homo_sapiens_OX_9606_GN_KCNQ3_PE_1_SV_2
Transcripts found: 7
✅ FOUND exact match!
ENST: ENST00000388996
Translation: ENSP00000373648
AA len: 872

Fetched CDS for matched KCNQ3 ENST: ENST00000388996 CDS len: 2619


hopefully this time

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install biopython requests

import requests
from Bio import SeqIO
from Bio.Seq import Seq

PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

ENST = {
    "KCNQ1": "ENST00000155840",
    "KCNQ2": "ENST00000359125",
    "KCNQ3": "ENST00000388996",   # ✅ FOUND
    "KCNQ4": "ENST00000347132",
    "KCNQ5": "ENST00000370398",
}

def ungap(s):
    return s.replace("-", "").replace(".", "")

def fetch_seq(enst_id, seq_type):
    url = f"https://rest.ensembl.org/sequence/id/{enst_id}?type={seq_type}"
    r = requests.get(url, headers={"Content-Type":"text/plain"})
    r.raise_for_status()
    return r.text.strip().replace("\n","").replace("\r","")

def translate_cds(nt):
    nt = "".join([c for c in nt.upper() if c in "ACGTN"])
    nt = nt[: (len(nt)//3)*3]
    aa = str(Seq(nt).translate(to_stop=False))
    if aa.endswith("*"):
        aa = aa[:-1]
    return aa

# Load target proteins from your protein MSA
targets = {}  # gene -> (msa_id, aa_ungapped)
for r in SeqIO.parse(PROT_MSA, "fasta"):
    for g in ENST.keys():
        if g in r.id:
            targets[g] = (r.id, ungap(str(r.seq)))
            break

print("Protein targets loaded from MSA:")
for g,(mid,aa) in targets.items():
    print(g, "len", len(aa))

# Fetch CDS, verify exact protein match, and write a 5-seq CDS fasta whose headers match MSA IDs
OUT_CDS_5 = "/content/kcnq_cds_5_matched.fasta"
with open(OUT_CDS_5, "w") as f:
    for g,enst in ENST.items():
        cds = fetch_seq(enst, "cds").upper()
        aa = translate_cds(cds)

        msa_id, target_aa = targets[g]
        print(f"\n{g}: ENST={enst} CDS_len={len(cds)} AA_len={len(aa)} target_len={len(target_aa)}")

        if aa != target_aa:
            print("❌ mismatch!")
            print("AA start:", aa[:40])
            print("Target :", target_aa[:40])
            raise RuntimeError(f"Stop: {g} ENST does not match the protein MSA target.")
        else:
            print("✅ Perfect match.")

        f.write(f">{msa_id}\n{cds}\n")

print("\nWrote:", OUT_CDS_5)
print("Preview:")
print("".join(open(OUT_CDS_5).read().splitlines(True)[:8]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Protein targets loaded from MSA:
KCNQ2 len 872
KCNQ4 len 695
KCNQ5 len 932
KCNQ3 len 872
KCNQ1 len 676

KCNQ1: ENST=ENST00000155840 CDS_len=2031 AA_len=676 target_len=676
✅ Perfect match.

KCNQ2: ENST=ENST00000359125 CDS_len=2619 AA_len=872 target_len=872
✅ Perfect match.

KCNQ3: ENST=ENST00000388996 CDS_len=2619 AA_len=872 target_len=872
✅ Perfect match.

KCNQ4: ENST=ENST00000347132 CDS_len=2088 AA_len=695 target_len=695
✅ Perfect match.

KCNQ5: ENST=ENST00000370398 CDS_len=2799 AA_len=932 target_len=932
✅ Perfect match.

Wrote: /content/kcnq_cds_5_matched.fasta
Preview:
>sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3
ATGGCCGCGGCCTCCTCCCCGCCCAGGGCCGAGAGGAAGCGCTGGGGTTGGGGCCGCCTGCCAGGCGCCCGGCGGGGCAGCGCGGGCCTGGCCAAGAAGTGCCCCTTCTCGCTGGAGCTGGCGGAGGGCGGCCCGGCGGGCGGCGCGCTCTACGCGCCCATCGCGCCCGGC

In [ ]:
!apt-get -qq update
!apt-get -qq install -y perl wget

# pal2nal v14
!wget -q -O /content/pal2nal.v14.tar.gz http://www.bork.embl.de/pal2nal/distribution/pal2nal.v14.tar.gz
!tar -xzf /content/pal2nal.v14.tar.gz -C /content

PROT_MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"
CODON_MSA = "/content/kcnq_codon_msa.fasta"

!perl /content/pal2nal.v14/pal2nal.pl "$PROT_MSA" "/content/kcnq_cds_5_matched.fasta" -output fasta > "$CODON_MSA"

from Bio import SeqIO
recs = list(SeqIO.parse(CODON_MSA, "fasta"))
print("Number of sequences:", len(recs))
lens = {len(r.seq) for r in recs}
print("Unique lengths:", lens)
print("All multiple of 3?", all(len(r.seq)%3==0 for r in recs))

print("\nPreview codon MSA:")
!head -n 10 "$CODON_MSA"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Number of sequences: 5
Unique lengths: {3360}
All multiple of 3? True

Preview codon MSA:
>sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2
ATGGTGCAGAAGTCGCGC------------------------------------------
AACGGC---------------GGCGTATACCCCGGCCCGAGCGGGGAGAAG---------
------------AAGCTGAAGGTGGGCTTCGTGGGG------------CTGGACCCCGGC
---------------------GCGCCCGACTCCACCCGGGACGGGGCGCTGCTGATC---
---------------------GCCGGCTCCGAGGCCCCCAAG------------------
---------CGCGGCAGCATCCTCAGCAAA---CCTCGCGCGGGCGGCGCG---------
------GGCGCCGGG---------------------------------------------
------------------------------------AAGCCCCCCAAGCGCAACGCCTTC
TACCGC---AAGCTGCAGAATTTCCTCTACAACGTGCTGGAGCGGCCGCGCGGCTGG---


# **BUILDING A TREE**

In [ ]:
!apt-get -qq update
!apt-get -qq install -y fasttree

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package fasttree.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../fasttree_2.1.11-2_amd64.deb ...
Unpacking fasttree (2.1.11-2) ...
Setting up fasttree (2.1.11-2) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
CODON_MSA="/content/kcnq_codon_msa.fasta"
TREE_OUT="/content/kcnq_tree_fasttree.nwk"

!FastTree -nt "$CODON_MSA" > "$TREE_OUT"
!cat "$TREE_OUT"

FastTree Version 2.1.11 Double precision (No SSE3)
Alignment: /content/kcnq_codon_msa.fasta
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Jukes-Cantor, CAT approximation with 20 rate categories
Initial topology in 0.00 seconds
Refining topology: 9 rounds ME-NNIs, 2 rounds ME-SPRs, 5 rounds ML-NNIs
Total branch-length 1.559 after 0.02 sec
ML-NNI round 1: LogLk = -14091.087 NNIs 1 max delta 3.22 Time 0.06
Switched to using 20 rate categories (CAT approximation)
Rate categories were divided by 0.730 so that average rate = 1.0
CAT-based log-likelihoods may not be comparable across runs
Use -gamma for approximate but comparable Gamma(20) log-likelihoods
ML-NNI round 2: LogLk = -13322.892 NNIs 0 max delta 0.00 Time 0.09
Turning off heuristics for final round of ML NNIs (converged)
ML-NNI round 3: LogLk = -13322.881 NNIs 0 max delta 0.00 Time 0.12

transforming into JSON

In [ ]:
!pip -q install biopython

from Bio import Phylo, SeqIO
from io import StringIO
import json

CODON_MSA = "/content/kcnq_codon_msa.fasta"
NEWICK = r"""(sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2:0.247223250,sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_3_OS_Homo_sapiens_OX_9606_GN_KCNQ3_PE_1_SV_2:0.369350284,(sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3:0.518011167,(sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_4_OS_Homo_sapiens_OX_9606_GN_KCNQ4_PE_1_SV_2:0.182027818,sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_5_OS_Homo_sapiens_OX_9606_GN_KCNQ5_PE_1_SV_3:0.384153347)0.999:0.108013187)0.759:0.038568964);"""

# 1) leaf order = FASTA order (critical)
leaf_ids = [r.id for r in SeqIO.parse(CODON_MSA, "fasta")]
N = len(leaf_ids)
leaf_label = {name: i+1 for i, name in enumerate(leaf_ids)}  # 1..N

print("Leaf order (FASTA):")
for i,name in enumerate(leaf_ids, start=1):
    print(i, name[:70]+"...")

# 2) parse tree
tree = Phylo.read(StringIO(NEWICK), "newick")

# FastTree sometimes outputs an unrooted tree; Bio.Phylo treats the given structure as rooted.
# We'll compute heights (time from leaves). Leaves = 0.
def height(clade):
    if clade.is_terminal():
        return 0.0
    hs = []
    for child in clade.clades:
        bl = child.branch_length or 0.0
        hs.append(height(child) + bl)
    return max(hs)

# 3) assign labels: leaves fixed by FASTA order; internal nodes by postorder
internal_label = {}
d1 = []
d2 = []
node_time = [None] * (2*N - 1)  # index label-1

next_internal = N + 1

def assign(clade):
    global next_internal
    if clade.is_terminal():
        lab = leaf_label[clade.name]
        node_time[lab-1] = 0.0
        return lab, 0.0

    # binary assumption (FastTree output here is binary)
    left, right = clade.clades
    labL, tL = assign(left)
    labR, tR = assign(right)

    # parent time = max(child_time + branch_length_to_child)
    blL = left.branch_length or 0.0
    blR = right.branch_length or 0.0
    tP = max(tL + blL, tR + blR)

    labP = next_internal
    next_internal += 1

    node_time[labP-1] = float(tP)
    d1.append(int(labL))
    d2.append(int(labR))
    return labP, tP

root_lab, root_t = assign(tree.root)
assert next_internal == 2*N, "Internal labeling count mismatch"

# 4) sanity checks (match what hello.c expects)
# daughters arrays length N-1, internal nodes labels N+1..2N-1 in the same order they were created
assert len(d1) == N-1 and len(d2) == N-1
assert all(x is not None for x in node_time)

OUT_JSON = "/content/kcnq_tree.json"
with open(OUT_JSON, "w") as f:
    json.dump({"node_time": node_time, "daughter_1": d1, "daughter_2": d2}, f, indent=2)

print("\nWrote:", OUT_JSON)
print("node_time (first 10):", node_time[:10])
print("daughter_1:", d1)
print("daughter_2:", d2)

Leaf order (FASTA):
1 sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
2 sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
3 sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
4 sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
5 sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...


ValueError: too many values to unpack (expected 2)

new attempt

In [ ]:
!pip -q install biopython

from Bio import Phylo, SeqIO
from io import StringIO
import json

CODON_MSA = "/content/kcnq_codon_msa.fasta"
NEWICK = r"""(sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_2_OS_Homo_sapiens_OX_9606_GN_KCNQ2_PE_1_SV_2:0.247223250,sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_3_OS_Homo_sapiens_OX_9606_GN_KCNQ3_PE_1_SV_2:0.369350284,(sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_1_OS_Homo_sapiens_OX_9606_GN_KCNQ1_PE_1_SV_3:0.518011167,(sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_4_OS_Homo_sapiens_OX_9606_GN_KCNQ4_PE_1_SV_2:0.182027818,sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_member_5_OS_Homo_sapiens_OX_9606_GN_KCNQ5_PE_1_SV_3:0.384153347)0.999:0.108013187)0.759:0.038568964);"""

GENES = ["KCNQ1","KCNQ2","KCNQ3","KCNQ4","KCNQ5"]

# 1) leaf order = FASTA order
leaf_ids = [r.id for r in SeqIO.parse(CODON_MSA, "fasta")]
N = len(leaf_ids)
leaf_label = {name: i+1 for i, name in enumerate(leaf_ids)}  # 1..N

print("Leaf order (FASTA):")
for i,name in enumerate(leaf_ids, start=1):
    print(i, name[:70]+"...")

tree = Phylo.read(StringIO(NEWICK), "newick")

# 2) resolve polytomies deterministically (turn any node with >2 children into a chain)
def resolve_polytomies(clade):
    # resolve recursively first
    for child in clade.clades:
        resolve_polytomies(child)

    # if >2 children, binarize by repeatedly grouping last two
    while len(clade.clades) > 2:
        a = clade.clades.pop()
        b = clade.clades.pop()
        # create a new internal clade with zero branch length to parent
        new = Phylo.Newick.Clade(branch_length=0.0)
        new.clades = [b, a]
        clade.clades.append(new)

resolve_polytomies(tree.root)

# optional: make output stable
tree.ladderize()

# 3) compute "height" (time from leaves); leaves time=0
def height(clade):
    if clade.is_terminal():
        return 0.0
    hs = []
    for child in clade.clades:
        bl = child.branch_length or 0.0
        hs.append(height(child) + bl)
    return max(hs) if hs else 0.0

# 4) assign labels: leaves fixed; internal nodes postorder N+1..2N-1
d1, d2 = [], []
node_time = [None] * (2*N - 1)
next_internal = N + 1

def assign(clade):
    global next_internal
    if clade.is_terminal():
        lab = leaf_label[clade.name]
        node_time[lab-1] = 0.0
        return lab, 0.0

    if len(clade.clades) != 2:
        raise RuntimeError(f"Still not binary at some node: {len(clade.clades)} children")

    left, right = clade.clades

    labL, tL = assign(left)
    labR, tR = assign(right)

    blL = left.branch_length or 0.0
    blR = right.branch_length or 0.0
    tP = max(tL + blL, tR + blR)

    labP = next_internal
    next_internal += 1

    node_time[labP-1] = float(tP)
    d1.append(int(labL))
    d2.append(int(labR))
    return labP, tP

root_lab, root_t = assign(tree.root)

assert next_internal == 2*N, f"Internal labeling mismatch: next_internal={next_internal}, expected={2*N}"
assert len(d1) == N-1 and len(d2) == N-1
assert all(x is not None for x in node_time)

OUT_JSON = "/content/kcnq_tree.json"
with open(OUT_JSON, "w") as f:
    json.dump({"node_time": node_time, "daughter_1": d1, "daughter_2": d2}, f, indent=2)

print("\nWrote:", OUT_JSON)
print("node_time:", node_time)
print("daughter_1:", d1)
print("daughter_2:", d2)

Leaf order (FASTA):
1 sp_O43526_KCNQ2_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
2 sp_P56696_KCNQ4_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
3 sp_Q9NR82_KCNQ5_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
4 sp_O43525_KCNQ3_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...
5 sp_P51787_KCNQ1_HUMAN_Potassium_voltage-gated_channel_subfamily_KQT_me...

Wrote: /content/kcnq_tree.json
node_time: [0.0, 0.0, 0.0, 0.0, 0.0, 0.384153347, 0.518011167, 0.556580131, 0.556580131]
daughter_1: [2, 5, 4, 1]
daughter_2: [3, 6, 7, 8]


# **NEW STEP**

In [ ]:
!git clone --recursive https://github.com/astheeggeggs/parsel.git
%cd parsel
!ls

Cloning into 'parsel'...
remote: Enumerating objects: 43, done.
remote: Total 43 (delta 0), reused 0 (delta 0), pack-reused 43 (from 1)
Receiving objects: 100% (43/43), 50.23 KiB | 886.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/parsel
amino_acids.c  libs		 load_from_json.h  README.md	       util.c
amino_acids.h  load_data.c	 Makefile	   selection_window.c  util.h
constants.c    load_data.h	 mcmc_moves.c	   selection_window.h
constants.h    load_from_json.c  mcmc_moves.h	   src


In [ ]:
!ls libs
!find libs -maxdepth 2 -type f -name "Makefile" | head

cJSON  Makefile
libs/Makefile


In [ ]:
!make clean
!make
!ls -l

rm -rf hello *.dSYM *.greg
gcc -O3   -Wall -Wextra -I libs/seq_file -I libs/string_buffer -I libs/cJSON -I libs/gsl-1.16 -Llibs/string_buffer -I . -o hello src/tools/hello.c amino_acids.c constants.c load_data.c load_from_json.c mcmc_moves.c selection_window.c util.c libs/cJSON/cJSON.c libs/gsl-1.16/.libs/libgsl.a -lstrbuf -lz -lm -lpthread 
src/tools/hello.c: In function ‘hello_parse_cmdline’:
src/tools/hello.c:136:17: warning: implicit declaration of function ‘sleep’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
  136 |                 sleep(1);
      |                 ^~~~~
load_data.c:7:10: fatal error: string_buffer.h: No such file or directory
    7 | #include "string_buffer.h"
      |          ^~~~~~~~~~~~~~~~~
compilation terminated.
load_from_json.c:7:10: fatal error: string_buffer.h: No such file or directory
    7 | #include "string_buffer.h"
      |          ^~~~~~~~~~~~~~~~~
compila

In [ ]:
!ls -R libs

libs:
cJSON  Makefile

libs/cJSON:
cJSON.c  cJSON.h  LICENSE  README  test.c  tests

libs/cJSON/tests:
test1  test2  test3  test4  test5


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libgsl-dev zlib1g-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libgslcblas0:amd64.
(Reading database ... 117541 files and directories currently installed.)
Preparing to unpack .../libgslcblas0_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgslcblas0:amd64 (2.7.1+dfsg-3) ...
Selecting previously unselected package libgsl27:amd64.
Preparing to unpack .../libgsl27_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgsl27:amd64 (2.7.1+dfsg-3) ...
Selecting previously unselected package libgsl-dev.
Preparing to unpack .../libgsl-dev_2.7.1+dfsg-3_amd64.deb ...
Unpacking libgsl-dev (2.7.1+dfsg-3) ...
Setting up libgslcblas0:amd64 (2.7.1+dfsg-3) ...
Setting up libgsl27:amd64 (2.7.1+dfsg-3) ...
Setting up libgsl-dev (2.7.1+dfsg-3) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.11) ...
/sbin/ldconfig.

In [ ]:
%cd /content/parsel
!grep -R --line-number "string_buffer" -n .
!grep -R --line-number "seq_file" -n .
!grep -R --line-number '#include "string_buffer.h"' -n .

/content/parsel
./load_data.c:7:#include "string_buffer.h"
./Makefile:6:INCLUDES=-I libs/seq_file -I libs/string_buffer -I libs/cJSON -I $(IDIR_GSL_HEADERS)
./Makefile:7:LIBDIRS=-Llibs/string_buffer
./load_from_json.c:7:#include "string_buffer.h"
./libs/Makefile:1:LIBS=seq_file string_buffer gsl cJSON
./libs/Makefile:7:	cd string_buffer && make clean
./libs/Makefile:12:string_buffer: string_buffer/Makefile
./libs/Makefile:13:	cd string_buffer && git pull && make
./libs/Makefile:24:string_buffer/Makefile:
./libs/Makefile:25:	git clone https://github.com/noporpoise/string_buffer.git
./load_data.c:8:#include "seq_file.h"
./load_data.c:74:  seq_file_t *file = seq_open(path);
./Makefile:6:INCLUDES=-I libs/seq_file -I libs/string_buffer -I libs/cJSON -I $(IDIR_GSL_HEADERS)
./libs/Makefile:1:LIBS=seq_file string_buffer gsl cJSON
./libs/Makefile:6:	cd seq_file && make clean
./libs/Makefile:9:seq_file: seq_file/Makefile
./libs/Makefile:10:	cd seq_file && git pull && make
./libs/Makefile:21:seq_

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libgsl-dev zlib1g-dev
%cd /content/parsel
!grep -R --line-number "string_buffer" .
!grep -R --line-number "strbuf" .
!grep -R --line-number "seq_file" .
!sed -n '1,220p' load_data.c
!sed -n '1,260p' load_from_json.c

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/content/parsel
./load_data.c:7:#include "string_buffer.h"
./Makefile:6:INCLUDES=-I libs/seq_file -I libs/string_buffer -I libs/cJSON -I $(IDIR_GSL_HEADERS)
./Makefile:7:LIBDIRS=-Llibs/string_buffer
./load_from_json.c:7:#include "string_buffer.h"
./libs/Makefile:1:LIBS=seq_file string_buffer gsl cJSON
./libs/Makefile:7:	cd string_buffer && make clean
./libs/Makefile:12:string_buffer: string_buffer/Makefile
./libs/Makefile:13:	cd string_buffer && git pull && make
./libs/Makefile:24:string_buffer/Makefile:
./libs/Makefile:25:	git clone https://github.com/noporpoise/string_buffer.git
./load_data.c:130:  strbuf_alloc(&line, 1024);
./load_data.c:135:  if(strbuf_readline(&line, fh) == 0) die("Empty CSV file: %s.", path);
./load_data.c:142:  for(i = 0; i < num_rows && strbuf_reset_readline(&line, fh); i++)


In [ ]:
%cd /content/parsel/libs
!make
!ls
%cd /content/parsel
!make clean
!make
!ls -l hello
!gzip -c /content/kcnq_tree.json > /content/kcnq_tree.json.gz
!ls -lh /content/kcnq_tree.json /content/kcnq_tree.json.gz
TREE="/content/kcnq_tree.json.gz"
CODON_MSA="/content/kcnq_codon_msa.fasta"
OUT="/content/kcnq_run_test"

%cd /content/parsel
!./hello "$TREE" "$CODON_MSA" "$OUT" -n 2000 -s 50 -w
!ls /content | grep kcnq_run_test
!head -n 30 /content/kcnq_run_test_log.txt 2>/dev/null || true
!head -n 30 /content/kcnq_run_test_sel.txt 2>/dev/null || true
!head -n 30 /content/kcnq_run_test_windows.txt 2>/dev/null || true

/content/parsel/libs
git clone https://github.com/noporpoise/seq_file.git
Cloning into 'seq_file'...
remote: Enumerating objects: 1027, done.
remote: Total 1027 (delta 0), reused 0 (delta 0), pack-reused 1027 (from 1)
Receiving objects: 100% (1027/1027), 294.96 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (617/617), done.
cd seq_file && git pull && make
Already up to date.
make[1]: Entering directory '/content/parsel/libs/seq_file'
mkdir -p bin
cc -Wall -Wextra -std=c99 -pedantic -I. -O3 -o bin/dnacat tools/dna_cat.c  -lpthread -lz -lm
In file included from /usr/include/x86_64-linux-gnu/bits/libc-header-start.h:33,
                 from /usr/include/stdlib.h:26,
                 from tools/dna_cat.c:11:
/usr/include/features.h:194:3: warning: #warning "_BSD_SOURCE and _SVID_SOURCE are deprecated, use _DEFAULT_SOURCE" []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wcpp-Wcpp]8;;]
  194 | # warning "_BSD_SOURCE and _SVID_SOURCE are deprecated, use _DEFAULT_SOURCE

new attempt

In [ ]:
%cd /content/parsel/libs
!make clean || true
!make
!ls -la

/content/parsel/libs
cd seq_file && make clean
make[1]: Entering directory '/content/parsel/libs/seq_file'
rm -rf bin
cd benchmarks; make clean
make[2]: Entering directory '/content/parsel/libs/seq_file/benchmarks'
rm -rf seqtest ktest *.dSYM *.greg
make[2]: Leaving directory '/content/parsel/libs/seq_file/benchmarks'
cd dev; make clean
make[2]: Entering directory '/content/parsel/libs/seq_file/dev'
rm -rf seqcmdline *.dSYM *.greg
make[2]: Leaving directory '/content/parsel/libs/seq_file/dev'
make[1]: Leaving directory '/content/parsel/libs/seq_file'
cd string_buffer && make clean
make[1]: Entering directory '/content/parsel/libs/string_buffer'
rm -rf string_buffer.o libstrbuf.a strbuf_test *.dSYM *.greg
rm -rf tmp.strbuf.*.txt tmp.strbuf.*.txt.gz
make[1]: Leaving directory '/content/parsel/libs/string_buffer'
cd seq_file && git pull && make
Already up to date.
make[1]: Entering directory '/content/parsel/libs/seq_file'
mkdir -p bin
cc -Wall -Wextra -std=c99 -pedantic -I. -O3 -o bin/dn

In [ ]:
!ls -la /content/parsel/libs/string_buffer || true
!find /content/parsel/libs/string_buffer -maxdepth 2 -type f \( -name "*strbuf*" -o -name "lib*.a" -o -name "lib*.so" \) || true

total 332
drwxr-xr-x 3 root root   4096 Jan 14 14:35 .
drwxr-xr-x 6 root root   4096 Jan 14 14:23 ..
drwxr-xr-x 8 root root   4096 Jan 14 14:35 .git
-rw-r--r-- 1 root root    122 Jan 14 14:23 .gitignore
-rw-r--r-- 1 root root  27036 Jan 14 14:35 libstrbuf.a
-rw-r--r-- 1 root root   6459 Jan 14 14:23 LICENSE
-rw-r--r-- 1 root root    871 Jan 14 14:23 Makefile
-rw-r--r-- 1 root root  14279 Jan 14 14:23 README.md
-rwxr-xr-x 1 root root 125400 Jan 14 14:35 strbuf_test
-rw-r--r-- 1 root root  41813 Jan 14 14:23 strbuf_test.c
-rw-r--r-- 1 root root  17738 Jan 14 14:23 stream_buffer.h
-rw-r--r-- 1 root root  25329 Jan 14 14:23 string_buffer.c
-rw-r--r-- 1 root root   9888 Jan 14 14:23 string_buffer.h
-rw-r--r-- 1 root root  25768 Jan 14 14:35 string_buffer.o
-rw-r--r-- 1 root root    186 Jan 14 14:23 .travis.yml
/content/parsel/libs/string_buffer/strbuf_test.c
/content/parsel/libs/string_buffer/libstrbuf.a
/content/parsel/libs/string_buffer/strbuf_test


In [ ]:
%cd /content/parsel
!make clean
!make
!ls -l hello

/content/parsel
rm -rf hello *.dSYM *.greg
gcc -O3   -Wall -Wextra -I libs/seq_file -I libs/string_buffer -I libs/cJSON -I libs/gsl-1.16 -Llibs/string_buffer -I . -o hello src/tools/hello.c amino_acids.c constants.c load_data.c load_from_json.c mcmc_moves.c selection_window.c util.c libs/cJSON/cJSON.c libs/gsl-1.16/.libs/libgsl.a -lstrbuf -lz -lm -lpthread 
src/tools/hello.c: In function ‘hello_parse_cmdline’:
src/tools/hello.c:136:17: warning: implicit declaration of function ‘sleep’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
  136 |                 sleep(1);
      |                 ^~~~~
mcmc_moves.c:70:14: warning: argument 9 of type ‘char[num_sites][num_sites]’ declared with 1 variable bound []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wvla-parameter-Wvla-parameter]8;;]
   70 |         char leaf_sequences[N][num_sites], int daughter_1[N-1], int daughter_2[N-1],
 

In [ ]:
%cd /content/parsel
!make clean >/dev/null 2>&1 || true
!make 2>&1 | tail -n 120

/content/parsel
   70 |         char leaf_sequences[N][num_sites], int daughter_1[N-1], int daughter_2[N-1],
      |         ~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~
In file included from mcmc_moves.c:9:
mcmc_moves.h:35:14: note: previously declared as ‘char[num_sites][N]’ with 2 variable bounds
   35 |         char leaf_sequences[num_sites][N],int daughter_1[N-1], int daughter_2[N-1],
      |         ~~~~~^~~~~~~~~~~~~~~~~~~~~~~~~~~~
mcmc_moves.c:70:48: warning: argument 10 of type ‘int[(int)N - 1]’ declared with mismatched bound ‘(int) N - 1’ [-Wvla-parameter]
   70 |         char leaf_sequences[N][num_sites], int daughter_1[N-1], int daughter_2[N-1],
      |                                            ~~~~^~~~~~~~~~~~~~~
In file included from mcmc_moves.c:9:
mcmc_moves.h:35:47: note: previously declared as ‘int[num_sites]’ with bound argument 5
   33 | void tree_likelihood(const int start_site, const int end_site, Decimal mu, Decimal indel_rate, const int num_sites,
      |                 

In [ ]:
%cd /content/parsel
!grep -R --line-number -E "^(int|long|unsigned|size_t)[[:space:]]+(indel_reject|indel_accept|mut_reject|mut_accept|wcoeff_reject|wcoeff_accept|split_reject|split_accept|grow_reject|grow_accept|merge_reject|merge_accept|sel_reject|sel_accept)[[:space:]]*;" .

/content/parsel


In [ ]:
%cd /content/parsel
!grep -R --line-number -E "\b(indel_reject|indel_accept|mut_reject|mut_accept|wcoeff_reject|wcoeff_accept|split_reject|split_accept|grow_reject|grow_accept|merge_reject|merge_accept|sel_reject|sel_accept)\b" .

/content/parsel
./src/tools/hello.c:457:	printf("selection accept: %i selection reject: %i\n", sel_accept, sel_reject);
./src/tools/hello.c:458:	printf("mutation accept: %i mutation reject: %i\n", mut_accept, mut_reject);
./src/tools/hello.c:459:	printf("indel accept: %i indel reject: %i\n", indel_accept, indel_reject);
./src/tools/hello.c:462:	printf("merge accept: %i merge reject: %i\n", merge_accept, merge_reject);
./src/tools/hello.c:463:	printf("split accept: %i split reject: %i\n", split_accept, split_reject);
./src/tools/hello.c:464:	printf("grow accept: %i grow reject: %i\n", grow_accept, grow_reject);
./src/tools/hello.c:465:	printf("window coeff accept: %i window coeff reject: %i\n", wcoeff_accept, wcoeff_reject);
./mcmc_moves.c:274:		sel_accept++;
./mcmc_moves.c:280:  		sel_reject++;
./mcmc_moves.c:327:		mut_accept++;
./mcmc_moves.c:338:		mut_reject++;
./mcmc_moves.c:379:		indel_accept++;
./mcmc_moves.c:390:		indel_reject++;
./mcmc_moves.c:533:				grow_reject++;
./mcmc_moves

In [ ]:
%cd /content/parsel
!grep -R --line-number -E "^\s*(int|long|unsigned|size_t)\s+.*\b(indel_reject|indel_accept|mut_reject|mut_accept|wcoeff_reject|wcoeff_accept|split_reject|split_accept|grow_reject|grow_accept|merge_reject|merge_accept|sel_reject|sel_accept)\b" .

/content/parsel
./mcmc_moves.h:17:int sel_accept, sel_reject;
./mcmc_moves.h:18:int merge_accept, merge_reject;
./mcmc_moves.h:19:int split_accept, split_reject;
./mcmc_moves.h:20:int grow_accept, grow_reject;
./mcmc_moves.h:21:int wcoeff_accept, wcoeff_reject;
./mcmc_moves.h:22:int mut_accept, mut_reject;
./mcmc_moves.h:23:int indel_accept, indel_reject;


In [ ]:
%cd /content/parsel
!grep -R --line-number -E "^\s*(int|long|unsigned|size_t)\s+[^;]*,(.*\b(indel_reject|indel_accept|mut_reject|mut_accept|wcoeff_reject|wcoeff_accept|split_reject|split_accept|grow_reject|grow_accept|merge_reject|merge_accept|sel_reject|sel_accept)\b)" .

/content/parsel
./mcmc_moves.h:17:int sel_accept, sel_reject;
./mcmc_moves.h:18:int merge_accept, merge_reject;
./mcmc_moves.h:19:int split_accept, split_reject;
./mcmc_moves.h:20:int grow_accept, grow_reject;
./mcmc_moves.h:21:int wcoeff_accept, wcoeff_reject;
./mcmc_moves.h:22:int mut_accept, mut_reject;
./mcmc_moves.h:23:int indel_accept, indel_reject;


In [ ]:
%cd /content/parsel
!for f in *.h src/tools/*.h src/*.h; do \
  if [ -f "$f" ]; then \
    sed -i -E 's/^([[:space:]]*)(int|long|unsigned|size_t)([[:space:]]+)(indel_reject|indel_accept|mut_reject|mut_accept|wcoeff_reject|wcoeff_accept|split_reject|split_accept|grow_reject|grow_accept|merge_reject|merge_accept|sel_reject|sel_accept)([[:space:]]*)(=.*)?;/\1extern \2\3\4\5;/' "$f"; \
  fi; \
done


/content/parsel


In [ ]:
%cd /content/parsel
!grep -q "BEGIN_MCMC_COUNTER_DEFS" util.c || cat >> util.c <<'EOF'

/* BEGIN_MCMC_COUNTER_DEFS */
int indel_reject = 0, indel_accept = 0;
int mut_reject = 0, mut_accept = 0;
int wcoeff_reject = 0, wcoeff_accept = 0;
int split_reject = 0, split_accept = 0;
int grow_reject = 0, grow_accept = 0;
int merge_reject = 0, merge_accept = 0;
int sel_reject = 0, sel_accept = 0;
/* END_MCMC_COUNTER_DEFS */
EOF


SyntaxError: invalid syntax (ipython-input-2943897801.py, line 4)

In [ ]:
%%bash
cd /content/parsel

# להוסיף את ההגדרות ל-util.c אם הן עדיין לא קיימות
grep -q "BEGIN_MCMC_COUNTER_DEFS" util.c || cat >> util.c <<'EOF'

/* BEGIN_MCMC_COUNTER_DEFS */
int indel_reject = 0, indel_accept = 0;
int mut_reject = 0, mut_accept = 0;
int wcoeff_reject = 0, wcoeff_accept = 0;
int split_reject = 0, split_accept = 0;
int grow_reject = 0, grow_accept = 0;
int merge_reject = 0, merge_accept = 0;
int sel_reject = 0, sel_accept = 0;
/* END_MCMC_COUNTER_DEFS */

EOF


In [ ]:
!tail -n 30 /content/parsel/util.c



  return str;
}

int discrete_sampling_dist(int n_bins, Decimal *p_bins)
{
  int i;
  double current_max = p_bins[0];
  double unif_sample = rand_lim(1);
  int current_bin = 0;
  
  for(i = 0; i < n_bins; i++) {
    if(unif_sample < current_max) break;
    current_bin++;
    current_max +=p_bins[i+1];
  }

  return current_bin;
}

/* BEGIN_MCMC_COUNTER_DEFS */
int indel_reject = 0, indel_accept = 0;
int mut_reject = 0, mut_accept = 0;
int wcoeff_reject = 0, wcoeff_accept = 0;
int split_reject = 0, split_accept = 0;
int grow_reject = 0, grow_accept = 0;
int merge_reject = 0, merge_accept = 0;
int sel_reject = 0, sel_accept = 0;
/* END_MCMC_COUNTER_DEFS */



In [ ]:
!cd /content/parsel && make clean && make


rm -rf hello *.dSYM *.greg
gcc -O3   -Wall -Wextra -I libs/seq_file -I libs/string_buffer -I libs/cJSON -I libs/gsl-1.16 -Llibs/string_buffer -I . -o hello src/tools/hello.c amino_acids.c constants.c load_data.c load_from_json.c mcmc_moves.c selection_window.c util.c libs/cJSON/cJSON.c libs/gsl-1.16/.libs/libgsl.a -lstrbuf -lz -lm -lpthread 
src/tools/hello.c: In function ‘hello_parse_cmdline’:
src/tools/hello.c:136:17: warning: implicit declaration of function ‘sleep’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
  136 |                 sleep(1);
      |                 ^~~~~
mcmc_moves.c:70:14: warning: argument 9 of type ‘char[num_sites][num_sites]’ declared with 1 variable bound []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wvla-parameter-Wvla-parameter]8;;]
   70 |         char leaf_sequences[N][num_sites], int daughter_1[N-1], int daughter_2[N-1],
      |         ~

In [ ]:
!ls -l /content/parsel/hello


ls: cannot access '/content/parsel/hello': No such file or directory


In [ ]:
%%bash
set -e
cd /content/parsel

NAMES="indel_reject indel_accept mut_reject mut_accept wcoeff_reject wcoeff_accept split_reject split_accept grow_reject grow_accept merge_reject merge_accept sel_reject sel_accept"

echo "==> Finding any remaining DEFINITIONS (before patch)"
for v in $NAMES; do
  grep -R --line-number -E "^[[:space:]]*(int|long|unsigned|size_t)[[:space:]]+[^;]*\\b${v}\\b[^;]*;" . || true
done

echo "==> Patching: convert any non-static, non-extern declarations/definitions to extern (in .c and .h)"
# Patch .c and .h files: if a line declares any of these globals and is not 'extern' or 'static', make it extern and drop any '= ...'
for f in $(find . -type f \( -name "*.c" -o -name "*.h" \)); do
  for v in $NAMES; do
    # Convert "int sel_accept = 0;" or "int sel_accept;" or "int a, sel_accept, b;"  -> "extern int ..."
    perl -0777 -pi -e "
      s/^([ \\t]*)(?!extern\\b)(?!static\\b)(int|long|unsigned|size_t)([ \\t]+[^;\\n]*\\b$v\\b[^;\\n]*)(?:=[^;\\n]*)?;/
        \$1extern \$2\$3;/mg
    " "$f"
  done
done

echo "==> Ensuring util.c has the ONE true definitions block"
grep -q "BEGIN_MCMC_COUNTER_DEFS" util.c || cat >> util.c <<'EOF'

/* BEGIN_MCMC_COUNTER_DEFS */
int indel_reject = 0, indel_accept = 0;
int mut_reject = 0, mut_accept = 0;
int wcoeff_reject = 0, wcoeff_accept = 0;
int split_reject = 0, split_accept = 0;
int grow_reject = 0, grow_accept = 0;
int merge_reject = 0, merge_accept = 0;
int sel_reject = 0, sel_accept = 0;
/* END_MCMC_COUNTER_DEFS */
EOF

echo "==> Rebuilding..."
make clean >/dev/null 2>&1 || true
make 2>&1 | tail -n 120


==> Finding any remaining DEFINITIONS (before patch)
./util.c:536:int indel_reject = 0, indel_accept = 0;
./mcmc_moves.h:23:int indel_accept, indel_reject;
./util.c:536:int indel_reject = 0, indel_accept = 0;
./mcmc_moves.h:23:int indel_accept, indel_reject;
./util.c:537:int mut_reject = 0, mut_accept = 0;
./mcmc_moves.h:22:int mut_accept, mut_reject;
./util.c:537:int mut_reject = 0, mut_accept = 0;
./mcmc_moves.h:22:int mut_accept, mut_reject;
./util.c:538:int wcoeff_reject = 0, wcoeff_accept = 0;
./mcmc_moves.h:21:int wcoeff_accept, wcoeff_reject;
./util.c:538:int wcoeff_reject = 0, wcoeff_accept = 0;
./mcmc_moves.h:21:int wcoeff_accept, wcoeff_reject;
./util.c:539:int split_reject = 0, split_accept = 0;
./mcmc_moves.h:19:int split_accept, split_reject;
./util.c:539:int split_reject = 0, split_accept = 0;
./mcmc_moves.h:19:int split_accept, split_reject;
./util.c:540:int grow_reject = 0, grow_accept = 0;
./mcmc_moves.h:20:int grow_accept, grow_reject;
./util.c:540:int grow_reject = 0

In [ ]:
!ls -l /content/parsel/hello

-rwxr-xr-x 1 root root 521680 Jan 14 14:51 /content/parsel/hello


some good values:

In [ ]:
TREE="/content/kcnq_tree.json.gz"
CODON_MSA="/content/kcnq_codon_msa.fasta"
OUT="/content/kcnq_run_test"

!cd /content/parsel && ./hello "$TREE" "$CODON_MSA" "$OUT" -n 2000 -s 50 -w
!ls /content | grep kcnq_run_test

argc - optind 3
optind 6
Using reversible-jump MCMC moves
Writing logs to: /content/kcnq_run_test/2026_1_14_14_53_15_log.txt
Writing summary to: /content/kcnq_run_test/2026_1_14_14_53_15_summary.txt
Loading sequences...
Loaded 5 sequences.
kappa:2.200000
phi:0.800000
Loaded 1120 sites.
consensus sequence:
ATGGTGCTCAAGTCGCGCAGGGCGGCGGGGGCGGCTGGCGGCGGCGACGACGGCGGCGCCGCCGGCCTCTGGGTGAAGAGCGGCGTAGCCGCGGCCCCGGCCGGCGCGAGGGCGGCGGCGGCCGACAACAAGCGCCTGGGCGTGGGCCTCCTGCCCGGCGCCCGGCGGGGGGACGCCGGCCGGGCCAAGCTGTTGCCGCTCTCGGCCGACCCCACCGAAGACGGCCCGCTGCTGCTGGGCACCCGCGCGGCCACGCAGGCGGGCGGCGGCGACGCCCCGCAGGAGAGCCGCCGGACCCCGCAGGGCGCCCGCATCAGCCTCCTGGGCAAGACCCCCCTCGCGCCCGGCGCGCCCCTTCCTGCGCCCGGCGCCGGCTCGGCCGCGCCCGCCGCGCCCCCAGTTGCCTCCGACCTTGGCCCGCGGCCGCCGGTGAGCCTAGACTCCCGCTCCGGCAACCACACCTCCCGCCGCAACGTCTTGTACCGCACCCGCCTGCAGAACTTCCTCTACAACGTGCTGGAGCGACCCCGCGGCTGGAAAGCGTTCATCTACCACGCCTTCGTGTTCCTCCTTGTCTTCGGCTGCCTGATTCTGTCTGTGCTGTCCACCATCCAGGAGTATGAGAAACTCGCCACAGGGTGTCTCTTCATCCTGGAGATCGTGATGATTGTGGTCTTTGGCTTGGAGTACATC

In [ ]:
!ls -lh /content/kcnq_run_test*

total 416K
-rw-r--r-- 1 root root 408K Jan 14 14:55 2026_1_14_14_53_15_log.txt
-rw-r--r-- 1 root root  100 Jan 14 14:55 2026_1_14_14_53_15_summary.txt


In [ ]:
!head -n 30 /content/kcnq_run_test_log.txt 2>/dev/null || true
!head -n 30 /content/kcnq_run_test_sel.txt 2>/dev/null || true
!head -n 30 /content/kcnq_run_test_windows.txt 2>/dev/null || true

In [ ]:
!ls -lah /content | grep -E "kcnq_run_test|run_test|kcnq_" || true
!ls -lah /content/kcnq_run_test* || true

-rw-r--r-- 1 root  root   13K Jan 14 13:43 kcnq_cds_5_matched.fasta
-rw-r--r-- 1 root  root   92K Jan 14 13:21 kcnq_cds_all.fasta
-rw-r--r-- 1 root  root   31K Jan 14 13:21 kcnq_cds_translated.fasta
-rw-r--r-- 1 root  root   18K Jan 14 13:43 kcnq_codon_msa.fasta
drwxr-xr-x 2 root  root  4.0K Jan 14 14:53 kcnq_run_test
-rw-r--r-- 1 root  root   701 Jan 14 13:47 kcnq_tree_fasttree.nwk
-rw-r--r-- 1 root  root   236 Jan 14 14:01 kcnq_tree.json
-rw-r--r-- 1 root  root   136 Jan 14 14:28 kcnq_tree.json.gz
total 424K
drwxr-xr-x 2 root root 4.0K Jan 14 14:53 .
drwxr-xr-x 1 root root 4.0K Jan 14 14:53 ..
-rw-r--r-- 1 root root 408K Jan 14 14:55 2026_1_14_14_53_15_log.txt
-rw-r--r-- 1 root root  100 Jan 14 14:55 2026_1_14_14_53_15_summary.txt


In [ ]:
!for f in /content/kcnq_run_test*; do echo "---- $f ----"; ls -lh "$f"; wc -l "$f"; done


---- <_io.TextIOWrapper name='/content/kcnq_tree.json' mode='w' encoding='utf-8'> ----
ls: cannot access '<_io.TextIOWrapper name='\''/content/kcnq_tree.json'\'' mode='\''w'\'' encoding='\''utf-8'\''>': No such file or directory
wc: '<_io.TextIOWrapper name='\''/content/kcnq_tree.json'\'' mode='\''w'\'' encoding='\''utf-8'\''>': No such file or directory


In [ ]:
!sed -n '1,60p' /content/kcnq_run_test_log.txt


sed: can't read /content/kcnq_run_test_log.txt: No such file or directory


In [ ]:
!ls -lah /content/kcnq_run_test* | cat


total 424K
drwxr-xr-x 2 root root 4.0K Jan 14 14:53 .
drwxr-xr-x 1 root root 4.0K Jan 14 14:53 ..
-rw-r--r-- 1 root root 408K Jan 14 14:55 2026_1_14_14_53_15_log.txt
-rw-r--r-- 1 root root  100 Jan 14 14:55 2026_1_14_14_53_15_summary.txt


In [ ]:
!sed -n '1,200p' /content/2026_1_14_14_53_15_summary.txt


sed: can't read /content/2026_1_14_14_53_15_summary.txt: No such file or directory


In [ ]:
!cd /content/parsel && ./hello -h 2>&1 | sed -n '1,200p'


./hello: invalid option -- 'h'
[src/tools/hello.c:101] Error: Unknown option: ?


In [ ]:
!ls -lah /content/kcnq_run_test

total 424K
drwxr-xr-x 2 root root 4.0K Jan 14 14:53 .
drwxr-xr-x 1 root root 4.0K Jan 14 14:53 ..
-rw-r--r-- 1 root root 408K Jan 14 14:55 2026_1_14_14_53_15_log.txt
-rw-r--r-- 1 root root  100 Jan 14 14:55 2026_1_14_14_53_15_summary.txt


In [ ]:
!sed -n '1,200p' /content/kcnq_run_test/2026_1_14_14_53_15_summary.txt

chain length: 2000
sample every: 50
log file path: /content/kcnq_run_test/2026_1_14_14_53_15_log.txt

In [ ]:
!tail -n 200 /content/kcnq_run_test/2026_1_14_14_53_15_summary.txt


chain length: 2000
sample every: 50
log file path: /content/kcnq_run_test/2026_1_14_14_53_15_log.txt

In [ ]:
!cd /content/parsel && ./hello -h 2>&1 | sed -n '1,220p'


./hello: invalid option -- 'h'
[src/tools/hello.c:101] Error: Unknown option: ?


something new

In [ ]:
!ls /content/parsel/src/tools


hello.c


In [ ]:
!grep -R "posterior" -n /content/parsel


another attempt

In [ ]:
!grep -n "Chain:" /content/parsel/src/tools/hello.c


399:			printf("Chain: %i\n", chain);


In [ ]:
!sed -n '360,460p' /content/parsel/src/tools/hello.c


	sel_mergesplit = (sel_mergesplit_on ? 7 : 0) + ind;
	sel_grow = (sel_grow_on ? 7 : 0) + sel_mergesplit;
	sel_wind_coeff = (sel_wind_coeff_on ? 7 : 0) + sel_grow;

	int which_move = sel_wind_coeff;
	int move;
	int sample_i = 1;

	// Burn in the mutation parameter first.
	int mutation_burn = 100; // DEV: add this as an option.
	for(chain=0; chain < mutation_burn; chain++)
	{
		mutation_move(&mu, indel_rate, num_sites, omega, gap_present, N, sequences_codons, daughter_1, daughter_2,
					&codon_to_codon, rate_out_of_codons, branch1, branch2, &codon_to_codon_tmp,
					&tree_site_L, &tmp_tree_site_L, &L);
	}

	fprintf(log_file, "state likelihood mu ");
	for(s = 0; s < num_sites; s++) {
		fprintf(log_file, "omega[%i] ", s);
	}
	fprintf(log_file, "\n");

	for (chain=0; chain < chain_length; chain++, sample_i++)
	{
		int choose_move = rand_lim(which_move);
		move = choose_move < sel ? 0 :
				(choose_move < mut ? 1 :
					(choose_move < ind ? 2 : 
						(choose_move < sel_mergesplit ? 3 :
			

# **THIS TIME IT'S GONNA WORK**

In [ ]:
import numpy as np

LOG = "/content/kcnq_run_test/2026_1_14_14_53_15_log.txt"

# נמצא את השורה שמתחילה ב-"state likelihood mu" ואז נתחיל לקרוא שורות דגימה
omegas = []
iters = []

with open(LOG, "r") as f:
    started = False
    for line in f:
        line = line.strip()
        if not started:
            if line.startswith("state likelihood mu"):
                started = True
            continue

        if not line:
            continue

        parts = line.split()
        # שורת דגימה אמורה להיות: chain, L, mu, ואז 1120 ערכי omega
        # נוודא שזה באמת מספרים
        try:
            int(parts[0])
            float(parts[1]); float(parts[2])
        except:
            continue

        iters.append(int(parts[0]))
        omegas.append([float(x) for x in parts[3:]])

omegas = np.array(omegas)

print("samples:", omegas.shape[0])
print("num_sites:", omegas.shape[1])
print("first iters:", iters[:5])
print("last iters:", iters[-5:])
print("omega min/median/max:", float(np.min(omegas)), float(np.median(omegas)), float(np.max(omegas)))


samples: 40
num_sites: 1120
first iters: [49, 99, 149, 199, 249]
last iters: [1799, 1849, 1899, 1949, 1999]
omega min/median/max: 0.001601 0.224018 256.540858


In [ ]:
q50 = np.quantile(omegas, 0.50, axis=0)   # 50%-CI
q95 = np.quantile(omegas, 0.95, axis=0)   # 95%-CI

print("site\t50.scn\t95.scn")
for i in range(10):
    print(i+1, q50[i], q95[i])

print("\n--- LAST 10 ---")
for i in range(1110, 1120):
    print(i+1, q50[i], q95[i])

site	50.scn	95.scn
1 0.38746 0.616494
2 0.38746 0.616494
3 0.38746 0.5933406
4 0.38746 0.592122
5 0.38746 0.592122
6 0.38746 0.592122
7 0.110499 0.592122
8 0.200888 0.592122
9 0.13976 0.592122
10 0.130724 0.592122

--- LAST 10 ---
1111 0.1551885 0.369777
1112 0.1551885 0.369777
1113 0.197375 0.369777
1114 0.197375 0.369777
1115 0.197375 0.369777
1116 0.197375 0.369777
1117 0.197375 0.369777
1118 0.197466 0.369777
1119 0.197466 0.369777
1120 0.197466 0.369777


In [ ]:
MSA = "/content/drive/My Drive/שנה ג/Project/KCNQ2/KCNQ MSA/kcnq_msa.fasta"

def read_fasta(path):
    seqs = {}
    cur = None
    buf = []
    with open(path) as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur is not None:
                    seqs[cur] = "".join(buf)
                cur = line[1:]
                buf = []
            else:
                buf.append(line)
        if cur is not None:
            seqs[cur] = "".join(buf)
    return seqs

seqs = read_fasta(MSA)

# נזהה את KCNQ2 (לפי הכותרת שלך)
kcnq2_id = [k for k in seqs.keys() if "KCNQ2" in k][0]
kcnq2_aln = seqs[kcnq2_id]
assert len(kcnq2_aln) == 1120, (len(kcnq2_aln), "alignment length mismatch")

kcnq2_pos = 0
rows = []
for msa_i, aa in enumerate(kcnq2_aln, start=1):  # msa_pos 1..1120
    if aa == "-":
        continue
    kcnq2_pos += 1
    rows.append((msa_i, kcnq2_pos, float(q50[msa_i-1]), float(q95[msa_i-1])))

print("Total KCNQ2 positions:", kcnq2_pos)
print("msa_pos\tKCNQ2_pos\t50.scn\t95.scn")
for r in rows[:20]:
    print(*r, sep="\t")

print("\n--- LAST 10 ---")
for r in rows[-10:]:
    print(*r, sep="\t")

Total KCNQ2 positions: 872
msa_pos	KCNQ2_pos	50.scn	95.scn
1	1	0.38746	0.616494
2	2	0.38746	0.616494
3	3	0.38746	0.5933406
4	4	0.38746	0.592122
5	5	0.38746	0.592122
6	6	0.38746	0.592122
21	7	0.160582	0.16321179999999985
22	8	0.160582	0.16321179999999985
28	9	0.1587445	0.280159
29	10	0.1587445	0.280159
30	11	0.397965	0.631104
31	12	0.397965	0.631104
32	13	0.397965	0.631104
33	14	0.397965	0.631104
34	15	0.397965	0.631104
35	16	0.397965	0.631104
36	17	0.397965	0.631104
37	18	0.397965	4.728633849999989
45	19	0.1840795	0.39334829999999993
46	20	0.4434525	0.820657

--- LAST 10 ---
1109	863	0.1551885	0.369777
1110	864	0.1551885	0.369777
1111	865	0.1551885	0.369777
1112	866	0.1551885	0.369777
1113	867	0.197375	0.369777
1114	868	0.197375	0.369777
1115	869	0.197375	0.369777
1116	870	0.197375	0.369777
1117	871	0.197375	0.369777
1118	872	0.197466	0.369777


In [ ]:
for msa_i, kpos, q50v, q95v in rows:
    print(f"{kpos}\t{q50v}\t{q95v}")

1	0.38746	0.616494
2	0.38746	0.616494
3	0.38746	0.5933406
4	0.38746	0.592122
5	0.38746	0.592122
6	0.38746	0.592122
7	0.160582	0.16321179999999985
8	0.160582	0.16321179999999985
9	0.1587445	0.280159
10	0.1587445	0.280159
11	0.397965	0.631104
12	0.397965	0.631104
13	0.397965	0.631104
14	0.397965	0.631104
15	0.397965	0.631104
16	0.397965	0.631104
17	0.397965	0.631104
18	0.397965	4.728633849999989
19	0.1840795	0.39334829999999993
20	0.4434525	0.820657
21	0.4434525	0.820657
22	0.419346	0.820657
23	0.370465	0.891236
24	0.370465	0.891236
25	0.490861	0.8940478499999999
26	0.085827	0.19787114999999997
27	0.1091	2.048205
28	0.236277	0.420175
29	0.236277	0.791881
30	0.385161	0.791881
31	0.3703455	0.546164
32	0.3703455	0.546164
33	0.3703455	0.546164
34	0.3703455	0.546164
35	0.3703455	0.546164
36	0.24939899999999998	0.254739
37	0.24939899999999998	0.254739
38	0.24939899999999998	0.254739
39	0.24939899999999998	0.254739
40	0.24939899999999998	0.254739
41	0.24939899999999998	0.254739
42	0.24939899999

In [ ]:
# header
print("KCNQ2_pos\t50.kcnq\t95.kcnq")

# rows
for msa_i, kpos, q50v, q95v in rows:
    print(f"{kpos}\t{q50v}\t{q95v}")

KCNQ2_pos	50.kcnq	95.kcnq
1	0.38746	0.616494
2	0.38746	0.616494
3	0.38746	0.5933406
4	0.38746	0.592122
5	0.38746	0.592122
6	0.38746	0.592122
7	0.160582	0.16321179999999985
8	0.160582	0.16321179999999985
9	0.1587445	0.280159
10	0.1587445	0.280159
11	0.397965	0.631104
12	0.397965	0.631104
13	0.397965	0.631104
14	0.397965	0.631104
15	0.397965	0.631104
16	0.397965	0.631104
17	0.397965	0.631104
18	0.397965	4.728633849999989
19	0.1840795	0.39334829999999993
20	0.4434525	0.820657
21	0.4434525	0.820657
22	0.419346	0.820657
23	0.370465	0.891236
24	0.370465	0.891236
25	0.490861	0.8940478499999999
26	0.085827	0.19787114999999997
27	0.1091	2.048205
28	0.236277	0.420175
29	0.236277	0.791881
30	0.385161	0.791881
31	0.3703455	0.546164
32	0.3703455	0.546164
33	0.3703455	0.546164
34	0.3703455	0.546164
35	0.3703455	0.546164
36	0.24939899999999998	0.254739
37	0.24939899999999998	0.254739
38	0.24939899999999998	0.254739
39	0.24939899999999998	0.254739
40	0.24939899999999998	0.254739
41	0.24939899999999998

In [ ]:
# positions you requested (duplicates kept on purpose)
positions = [
    1,106,107,113,114,126,127,130,141,144,144,144,153,175,178,185,192,194,196,198,
    201,201,203,205,207,213,213,214,239,247,253,254,254,256,258,265,265,265,265,
    267,268,268,268,268,269,272,274,274,276,277,277,281,284,285,285,287,290,290,
    294,294,295,296,301,304,306,309,313,315,317,318,325,333,335,337,339,341,344,
    352,353,355,358,359,362,386,489,541,541,546,546,547,553,553,554,556,556,558,
    560,563,567,581,581,588,592,637,764
]

# build lookup from the full rows
# rows is assumed to be: [(msa_i, kpos, q50v, q95v), ...]
vals = {kpos: (q50v, q95v) for msa_i, kpos, q50v, q95v in rows}

# print tab-separated for easy Excel paste
print("KCNQ2_pos\t50.kcnq\t95.kcnq")
for p in positions:
    q50v, q95v = vals[p]
    print(f"{p}\t{q50v}\t{q95v}")

KCNQ2_pos	50.kcnq	95.kcnq
1	0.38746	0.616494
106	0.187744	0.22708414999999998
107	0.187744	0.22708414999999998
113	0.187744	0.22708414999999998
114	0.187744	0.22708414999999998
126	0.187744	0.336016
127	0.21560649999999998	0.336016
130	0.21560649999999998	0.29194
141	0.12755450000000002	0.322521
144	0.12755450000000002	0.322521
144	0.12755450000000002	0.322521
144	0.12755450000000002	0.322521
153	0.0446035	0.214385
175	0.351521	0.8199516999999997
178	0.351521	0.8199516999999997
185	0.351521	0.8199516999999997
192	0.201875	0.8199516999999997
194	0.19991	0.8199516999999997
196	0.19991	0.8199516999999997
198	0.19991	0.8199516999999997
201	0.19991	0.8199516999999997
201	0.19991	0.8199516999999997
203	0.170777	0.8199516999999997
205	0.170777	0.8199516999999997
207	0.170777	0.8199516999999997
213	0.303207	0.8199516999999997
213	0.303207	0.8199516999999997
214	0.303207	0.8199516999999997
239	0.28152699999999997	0.8199516999999997
247	0.150812	0.8199516999999997
253	0.252077	5.688449
254	0.177